# 1. Environment Setup

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# 1. Install necessary packages quietly
print("Installing dependencies...")
packages = ["pandas", "numpy", "scikit-learn", "requests", "tqdm", "pyarrow"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)

# 2. Create Project Root in Colab's temporary storage (Fast!)
ROOT = Path("/content/evidvar")
ROOT.mkdir(parents=True, exist_ok=True)

# Subfolders
(ROOT / "data/raw").mkdir(exist_ok=True)
(ROOT / "data/interim").mkdir(exist_ok=True)
(ROOT / "data/processed").mkdir(exist_ok=True)
(ROOT / "results/tables").mkdir(exist_ok=True)
(ROOT / "results/figures").mkdir(exist_ok=True)
(ROOT / "src").mkdir(exist_ok=True)

print(f"✅ Environment ready. Project root: {ROOT}")

# 2. Download ClinVar and Extract BRCA1/BRCA2 Variants with rsIDs

In [ ]:
import pandas as pd
from pathlib import Path
import requests

ROOT = Path("/content/evidvar")
RAW_FILE = ROOT / "data/raw/clinvar_variant_summary.txt.gz"

# 1. Download ClinVar if not present
if not RAW_FILE.exists():
    print("Downloading ClinVar summary (~435MB)... This may take 1-2 mins.")
    url = "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz"
    # Using wget is more robust for large files in Linux environments like Colab
    !wget -O "{RAW_FILE}" "{url}"
else:
    print("ClinVar file already exists. Skipping download.")

# 2. Inspect Headers to find the rsID column name dynamically
print("\nInspecting headers...")
header_df = pd.read_csv(RAW_FILE, sep="\t", compression="gzip", nrows=1)
cols = header_df.columns.tolist()

# Common names for rsID in ClinVar: 'RS', 'dbSNP_ID', 'Reference SNP cluster ID'
rs_col_candidates = [c for c in cols if 'rs' in c.lower() or 'dbsnp' in c.lower() or 'reference snp' in c.lower()]
rs_col_name = rs_col_candidates[0] if rs_col_candidates else None

if not rs_col_name:
    raise ValueError(f"Could not find rsID column. Available columns: {cols[:10]}...")

print(f"Detected rsID column: '{rs_col_name}'")

# 3. Load ONLY necessary columns to save RAM
target_cols = ["VariationID", "GeneSymbol", "ClinicalSignificance", "ReviewStatus", rs_col_name]
# Ensure no duplicates
target_cols = list(dict.fromkeys(target_cols))

print(f"Loading data with columns: {target_cols}...")
df_full = pd.read_csv(
    RAW_FILE,
    sep="\t",
    compression="gzip",
    low_memory=False,
    usecols=target_cols
)

print(f"Loaded shape: {df_full.shape}")

# 4. Filter for BRCA1/BRCA2
mask_gene = df_full['GeneSymbol'].astype(str).str.contains(r'\bBRCA[12]\b', case=False, regex=True, na=False)
brca_df = df_full[mask_gene].copy()
print(f"BRCA rows found: {len(brca_df):,}")

# 5. Clean rsIDs
def clean_rs(val):
    s = str(val).strip().lower()
    if s.startswith('rs'): return s
    if s.isdigit(): return f"rs{s}"
    return None

brca_df['CleanRSID'] = brca_df[rs_col_name].apply(clean_rs)
brca_df = brca_df[brca_df['CleanRSID'].notna()]
print(f"Valid BRCA variants with rsIDs: {len(brca_df):,}")

# 6. Save Intermediate File
INTERIM_FILE = ROOT / "data/interim/brca_with_rsid.csv.gz"
brca_df.to_csv(INTERIM_FILE, index=False, compression='gzip')
print(f"✅ Saved intermediate data to: {INTERIM_FILE}")

# 3. Label Cleaning and Stratified Train/Test Split

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pathlib import Path

ROOT = Path("/content/evidvar")
INPUT_FILE = ROOT / "data/interim/brca_with_rsid.csv.gz"

print("Loading intermediate data...")
df = pd.read_csv(INPUT_FILE)
print(f"Shape: {df.shape}")

# 1. Map Labels
def map_label(sig):
    s = str(sig).lower()
    # Pathogenic Class (1)
    if 'pathogenic' in s and 'likely' not in s and 'conflicting' not in s: return 1
    if 'likely pathogenic' in s: return 1
    # Benign Class (0)
    if 'benign' in s and 'likely' not in s and 'conflicting' not in s: return 0
    if 'likely benign' in s: return 0
    # Uncertain/Other (NaN)
    return np.nan

df['label'] = df['ClinicalSignificance'].apply(map_label)

# 2. Quality Filter
acceptable_reviews = [
    "criteria provided, multiple submitters, no conflicts",
    "reviewed by expert panel",
    "criteria provided, single submitter"
]
quality_mask = df['ReviewStatus'].isin(acceptable_reviews)
labeled_mask = df['label'].isin([0, 1])

# Keep only high-quality, labeled variants
clean_pool = df[labeled_mask & quality_mask].drop_duplicates(subset=['VariationID'])
print(f"High-quality unique variants for modeling: {len(clean_pool):,}")

# 3. Stratified Split (80% Train+Val, 20% Test)
X_temp, X_test = train_test_split(
    clean_pool, test_size=0.2, random_state=42, stratify=clean_pool['label']
)

# Split Temp into Train (70%) and Val (30%)
X_train, X_val = train_test_split(
    X_temp, test_size=0.3, random_state=42, stratify=X_temp['label']
)

print(f"Train size: {len(X_train)}, Val size: {len(X_val)}, Test size: {len(X_test)}")

# 4. Save Splits
OUT_DIR = ROOT / "data/processed"
X_train.to_csv(OUT_DIR / "train_set.csv.gz", index=False, compression='gzip')
X_val.to_csv(OUT_DIR / "val_set.csv.gz", index=False, compression='gzip')
X_test.to_csv(OUT_DIR / "test_set.csv.gz", index=False, compression='gzip')

print("✅ Saved Train/Val/Test sets.")

# 4. Ensembl VEP Annotation

In [ ]:
import requests
import json
import time
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from tenacity import retry, stop_after_attempt, wait_exponential, RetryError

# Install tenacity if not present (lightweight library for retries)
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tenacity"])

ROOT = Path("/content/evidvar")
CACHE_FILE = ROOT / "data/interim/ensembl_cache.json"

# 1. Load Datasets
train_df = pd.read_csv(ROOT / "data/processed/train_set.csv.gz")
val_df = pd.read_csv(ROOT / "data/processed/val_set.csv.gz")
test_df = pd.read_csv(ROOT / "data/processed/test_set.csv.gz")

all_dfs = pd.concat([train_df, val_df, test_df], ignore_index=True)
unique_rsids = all_dfs['CleanRSID'].unique().tolist()
print(f"Total unique rsIDs to annotate: {len(unique_rsids):,}")

# 2. Initialize Cache (Load existing progress)
cached_results = {}
if CACHE_FILE.exists():
    with open(CACHE_FILE, 'r') as f:
        cached_results = json.load(f)
    print(f"Loaded {len(cached_results)} from existing cache.")

ids_to_fetch = [rid for rid in unique_rsids if rid not in cached_results]
print(f"Remaining to fetch: {len(ids_to_fetch):,}")

if len(ids_to_fetch) == 0:
    print("All IDs already cached! Skipping fetch.")
else:
    # 3. Configure API Settings
    ENSEMBL_URL = "https://rest.ensembl.org/vep/human/id/"
    HEADERS = {"Content-Type": "application/json", "Accept": "application/json"}
    BATCH_SIZE = 50  # Reduced batch size to prevent timeouts
    DELAY_SECONDS = 1.0 # Increased delay between batches

    # Helper function with retry logic
    @retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
    def fetch_batch(batch_ids):
        payload = {"ids": batch_ids}
        resp = requests.post(
            ENSEMBL_URL,
            headers=HEADERS,
            data=json.dumps(payload),
            timeout=60 # Increased timeout
        )
        resp.raise_for_status() # Raise exception for bad status codes
        return resp.json()

    print("Starting robust batch fetching...")
    pbar = tqdm(range(0, len(ids_to_fetch), BATCH_SIZE))

    for i in pbar:
        batch_ids = ids_to_fetch[i:i+BATCH_SIZE]

        try:
            data = fetch_batch(batch_ids)

            # Update cache
            for item in data:
                input_id = item.get('input')
                if input_id:
                    cached_results[input_id] = item

            # Save cache periodically (every 10 batches) to avoid frequent disk writes
            if (i // BATCH_SIZE) % 10 == 0:
                with open(CACHE_FILE, 'w') as f:
                    json.dump(cached_results, f)

        except RetryError as e:
            print(f"\nFailed to fetch batch starting at index {i} after retries. Error: {e}")
            # Continue to next batch instead of crashing entire script
            continue
        except Exception as e:
            print(f"\nUnexpected error in batch {i}: {e}")
            continue

        time.sleep(DELAY_SECONDS)

    # Final save of cache
    with open(CACHE_FILE, 'w') as f:
        json.dump(cached_results, f)
    print("Fetching complete. Cache saved.")

# 4. Parse Results into DataFrame
def parse_vep(res_obj):
    if not res_obj:
        return {'Consequence': 'NA', 'SIFT': np.nan, 'PolyPhen': np.nan, 'AF': np.nan}

    tcs = res_obj.get('transcript_consequences', [])
    canonical = None
    for tc in tcs:
        if tc.get('canonical'):
            canonical = tc
            break
    if not canonical and tcs:
        canonical = tcs[0]

    cons = "unknown"
    sift = np.nan
    polyphen = np.nan

    if canonical:
        terms = canonical.get('consequence_terms', [])
        if terms: cons = ";".join(terms)
        sift = canonical.get('sift_score', np.nan)
        polyphen = canonical.get('polyphen_score', np.nan)

    af = np.nan
    colocated = res_obj.get('colocated_variants', [])
    if colocated:
        for cv in colocated:
            freqs = cv.get('frequencies', {})
            if freqs:
                vals = []
                for k,v in freqs.items():
                    if isinstance(v, dict) and 'af' in v: vals.append(float(v['af']))
                    elif isinstance(v, float): vals.append(v)
                if vals: af = sum(vals)/len(vals)
                break

    return {'Consequence': cons, 'SIFT': sift, 'PolyPhen': polyphen, 'AF': af}

features_list = []
for _, row in all_dfs.iterrows():
    rid = row['CleanRSID']
    res = cached_results.get(rid)
    feats = parse_vep(res)
    features_list.append(feats)

feat_df = pd.DataFrame(features_list)
annotated_df = pd.concat([all_dfs.reset_index(drop=True), feat_df], axis=1)

# 5. Save Final Annotated Dataset
FINAL_OUT = ROOT / "data/processed/clinvar_brca_annotated_final.csv.gz"
annotated_df.to_csv(FINAL_OUT, index=False, compression='gzip')

print(f"\n✅ Saved final annotated dataset: {FINAL_OUT}")
print("\nSample Data:")
print(annotated_df[['CleanRSID', 'label', 'Consequence', 'SIFT', 'PolyPhen', 'AF']].head())

print("\nNull counts (How many variants missing features?):")
print(annotated_df[['SIFT', 'PolyPhen', 'AF']].isnull().sum())

# 5. Feature Engineering Rationale
SIFT and PolyPhen are only defined for missense variants. Frameshift, stop-gained, splice-site, intronic, and synonymous variants often have missing substitution scores. Dropping these rows would bias the dataset toward missense variants and remove many pathogenic truncating variants. Therefore, we use consequence-class flags and a model that handles missing values natively.

# 6. Structural Baseline Model

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
import joblib
from pathlib import Path

ROOT = Path("/content/evidvar")
DATA_FILE = ROOT / "data/processed/clinvar_brca_annotated_final.csv.gz"

print("Loading annotated data...")
df = pd.read_csv(DATA_FILE)

# 1. FEATURE ENGINEERING
consequence_types = df['Consequence'].fillna('unknown').str.split(';')

df['is_frameshift'] = consequence_types.apply(lambda x: 1 if 'frameshift' in x else 0)
df['is_stop_gained'] = consequence_types.apply(lambda x: 1 if 'stop_gained' in x else 0)
df['is_splice_site'] = consequence_types.apply(lambda x: 1 if 'splice_acceptor' in x or 'splice_donor' in x else 0)
df['is_misSense'] = consequence_types.apply(lambda x: 1 if 'missense' in x else 0)
df['is_intron'] = consequence_types.apply(lambda x: 1 if 'intron' in x else 0)
df['is_synonymous'] = consequence_types.apply(lambda x: 1 if 'synonymous' in x else 0)

numeric_cols = ['SIFT', 'PolyPhen', 'AF']
feature_cols = [
    'is_frameshift', 'is_stop_gained', 'is_splice_site',
    'is_misSense', 'is_intron', 'is_synonymous'
] + numeric_cols

X = df[feature_cols]
y = df['label']

valid_mask = y.notna()
X = X[valid_mask]
y = y[valid_mask]

print(f"Final Dataset Shape: {X.shape}")

# 2. TRAIN/TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. MODEL TRAINING
model = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    min_samples_leaf=20,
    random_state=42
)

print("\nTraining Baseline Model...")
model.fit(X_train, y_train)

# 4. EVALUATION
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba > 0.5).astype(int)

auc_roc = roc_auc_score(y_test, y_pred_proba)
acc = accuracy_score(y_test, y_pred)

print(f"\n--- Baseline Performance (Structured Features Only) ---")
print(f"AUROC: {auc_roc:.4f}")
print(f"Accuracy: {acc:.4f}")

# 5. SAVE MODEL & METRICS (FIXED SYNTAX)
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(model, MODEL_DIR / "baseline_structural_model.pkl")

# Corrected DataFrame creation
metrics_df = pd.DataFrame({
    'Metric': ['AUROC', 'Accuracy'],
    'Value': [float(auc_roc), float(acc)], # Ensure floats
    'Model_Type': ['Structural_Baseline', 'Structural_Baseline']
})

metrics_df.to_csv(ROOT / "results/tables/baseline_metrics.csv", index=False)

print(f"\n✅ Saved baseline model to: {MODEL_DIR}/baseline_structural_model.pkl")
print(f"✅ Saved metrics to: {ROOT}/results/tables/baseline_metrics.csv")

# 7. PubMed Abstract Retrieval

In [ ]:
import requests
import xml.etree.ElementTree as ET
import time
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

ROOT = Path("/content/evidvar")
PUBMED_RAW_XML = ROOT / "data/interim/pubmed_raw.xml" # Optional backup
PUBMED_PROCESSED_CSV = ROOT / "data/interim/pubmed_abstracts.csv.gz"

# Check if we already processed this
if PUBMED_PROCESSED_CSV.exists():
    print("✅ PubMed abstracts already exist. Skipping fetch.")
    df_pubs = pd.read_csv(PUBMED_PROCESSED_CSV)
else:
    # 1. CONFIGURATION
    SEARCH_TERM = "(BRCA1[Title/Abstract] OR BRCA2[Title/Abstract]) AND (pathogenic OR mutation OR variant)"
    RETMAX = 5000  # Number of papers to retrieve

    BASE_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"

    print(f"Searching PubMed for: {SEARCH_TERM}")
    print(f"Limit: {RETMAX} papers...")

    # 2. ESSEARCH (Get IDs)
    params_search = {
        "db": "pubmed",
        "term": SEARCH_TERM,
        "retmax": RETMAX,
        "sort": "relevance"
    }

    resp_search = requests.get(BASE_URL + "esearch.fcgi", params=params_search)
    root_search = ET.fromstring(resp_search.content)
    id_list = [elem.text for elem in root_search.findall('.//Id')]

    print(f"Found {len(id_list)} PMIDs.")

    if len(id_list) == 0:
        raise ValueError("No PMIDs found. Check internet connection or search term.")

    # 3. EFETCH (Get Abstracts in Batches)
    BATCH_SIZE = 200
    abstracts_data = []

    print("Fetching abstracts...")
    # Use tqdm for progress bar
    pbar = tqdm(range(0, len(id_list), BATCH_SIZE))

    for i in pbar:
        batch_ids = id_list[i:i+BATCH_SIZE]

        params_fetch = {
            "db": "pubmed",
            "id": ",".join(batch_ids),
            "rettype": "xml",
            "retmode": "text"
        }

        try:
            resp_fetch = requests.get(BASE_URL + "efetch.fcgi", params=params_fetch, timeout=30)
            resp_fetch.raise_for_status()

            root_fetch = ET.fromstring(resp_fetch.content)

            for article in root_fetch.findall('.//PubmedArticle'):
                pmid_elem = article.find('.//PMID')
                title_elem = article.find('.//ArticleTitle')

                pmid = pmid_elem.text if pmid_elem is not None else ""
                title = "".join(title_elem.itertext()).strip() if title_elem is not None else ""

                # Extract Abstract Text (handle multiple sections if any)
                abstract_parts = []
                for abs_item in article.findall('.//AbstractText'):
                    abstract_parts.append("".join(abs_item.itertext()))

                abstract_text = " ".join(abstract_parts).strip()

                if abstract_text: # Only keep if there is an actual abstract
                    abstracts_data.append({
                        "PMID": pmid,
                        "Title": title,
                        "Abstract": abstract_text
                    })

        except Exception as e:
            print(f"\nError fetching batch starting at index {i}: {e}")
            continue

        time.sleep(0.4) # Polite delay between batches (NCBI limit: 3 req/sec without key)

    # 4. SAVE TO DISK
    df_pubs = pd.DataFrame(abstracts_data)
    df_pubs.to_csv(PUBMED_PROCESSED_CSV, index=False, compression='gzip')

    print(f"\n✅ Saved {len(df_pubs)} abstracts to {PUBMED_PROCESSED_CSV}")

# Display Sample
print("\nSample Data:")
display(df_pubs.head())
print(f"\nTotal Papers Retrieved: {len(df_pubs):,}")

# 8. Strict HGVS Evidence Mining

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from collections import defaultdict, Counter
from tqdm.auto import tqdm

ROOT = Path("/content/evidvar")
RAW_CLINVAR = ROOT / "data/raw/clinvar_variant_summary.txt.gz"
ABSTRACTS_FILE = ROOT / "data/interim/pubmed_abstracts.csv.gz"

if not RAW_CLINVAR.exists():
    print("Raw ClinVar missing. Downloading...")
    !wget -q -O "{RAW_CLINVAR}" "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz"
if not ABSTRACTS_FILE.exists():
    raise FileNotFoundError("pubmed_abstracts.csv.gz missing. Re-run the PubMed fetch cell.")

# ---- STRICT HGVS regex: capture the FULL change incl. + - > * / alleles ----
RE_C = re.compile(r"c\.\d[\w+\-/>*_]*")            # c.631G>T, c.631+1G>A, c.68_69delAG, c.5266dupC
RE_P = re.compile(r"p\.[A-Z][a-z]{2}\d+[A-Za-z0-9*/_]*")  # p.Gly211Ter, p.Glu23Valfs, p.Gln178*

def is_specific(tok):
    """Reject bare positions like 'c.631' or 'p.631'. A real allele change must
    contain a non-digit, non-underscore character after the position number."""
    if not isinstance(tok, str) or len(tok) < 5:
        return False
    body = tok[2:]                      # drop 'c.' / 'p.'
    rest = re.sub(r"^[0-9_]+", "", body)  # drop leading digits/underscores
    return bool(re.search(r"[A-Za-z+*/>\-]", rest)) and len(rest) >= 1

def extract_hgvs(text):
    if not isinstance(text, str):
        return []
    toks = RE_C.findall(text) + RE_P.findall(text)
    out = []
    for t in toks:
        t = t.strip().rstrip(".,;:)")
        if is_specific(t):
            out.append(t.lower())
    return out

# =========================================================
# STEP 1: Build SPECIFIC HGVS -> VariationID map (chunked, memory-safe)
# =========================================================
print("STEP 1: Building strict HGVS token dictionary from ClinVar (chunked)...")

token_to_vids = defaultdict(set)
brca_rows = 0
chunk_iter = pd.read_csv(
    RAW_CLINVAR, sep="\t", compression="gzip", low_memory=False,
    usecols=["VariationID", "GeneSymbol", "Name"],
    chunksize=400000
)
for chunk in tqdm(chunk_iter, desc="ClinVar chunks"):
    mask = chunk["GeneSymbol"].astype(str).str.contains(r"\bBRCA[12]\b", case=False, regex=True, na=False)
    sub = chunk[mask]
    if len(sub) == 0:
        continue
    brca_rows += len(sub)
    for vid, name in zip(sub["VariationID"].astype(str), sub["Name"]):
        for tok in set(extract_hgvs(name)):      # dedupe within a record
            token_to_vids[tok].add(vid)

print(f"   Scanned {brca_rows:,} BRCA rows.")
print(f"   Strict unique tokens: {len(token_to_vids):,}")

# Diagnostic: how many tokens map to >1 variant? (collisions)
multi = {t: v for t, v in token_to_vids.items() if len(v) > 1}
print(f"   Tokens shared by >1 variant: {len(multi):,} (these are genuine co-occurrences, e.g. same allele in two records)")
# Show the worst offenders so we can eyeball them
worst = sorted(multi.items(), key=lambda kv: -len(kv[1]))[:5]
print("   Top shared tokens:", [(t, len(v)) for t, v in worst])

# =========================================================
# STEP 2: Mine evidence sentences with STRICT matching
# =========================================================
print("\nSTEP 2: Mining evidence sentences (strict)...")
df_pubs = pd.read_csv(ABSTRACTS_FILE)
SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")

evidence_rows = []
seen = set()
for _, row in tqdm(df_pubs.iterrows(), total=len(df_pubs), desc="Abstracts"):
    pmid = str(row["PMID"])
    text = f"{row.get('Title','')} {row.get('Abstract','')}"
    if not isinstance(text, str) or len(text) < 20:
        continue
    for sent in SENT_SPLIT.split(text):
        sent = sent.strip()
        if len(sent) < 20:
            continue
        toks = extract_hgvs(sent)
        if not toks:
            continue
        for t in toks:
            vids = token_to_vids.get(t)
            if not vids:
                continue
            for vid in vids:
                key = (vid, pmid, sent)
                if key in seen:
                    continue
                seen.add(key)
                evidence_rows.append({
                    "VariationID": vid, "HGVS_Token": t,
                    "PMID": pmid, "Sentence": sent
                })

df_ev = pd.DataFrame(evidence_rows)
print(f"\n   Extracted {len(df_ev):,} evidence sentences.")
if len(df_ev):
    print(f"   Covering {df_ev['VariationID'].nunique():,} distinct variants.")

    # ---- KEY DIAGNOSTIC: variants-per-sentence (the old bug exploded this) ----
    vps = df_ev.groupby(["PMID", "Sentence"])["VariationID"].nunique()
    print(f"\n   Variants-per-sentence distribution:")
    print(vps.value_counts().sort_index().head(10).to_string())
    print(f"   Max variants on a single sentence: {vps.max()}")
    # Show the most 'crowded' sentences so we can confirm they are REAL co-mentions
    crowded = (df_ev.assign(n=vps.reset_index().set_index(['PMID','Sentence'])['VariationID'].reindex(
                  df_ev.set_index(['PMID','Sentence']).index).values)
               .sort_values('n', ascending=False).drop_duplicates(['PMID','Sentence']).head(3))
    print("\n   Most multi-variant sentences (verify these genuinely name several alleles):")
    for _, r in crowded.iterrows():
        print(f"     [{r['n']} vars] {r['Sentence'][:160]}...")

    # ---- Attach labels + features ----
    df_ann = pd.read_csv(ROOT / "data/processed/clinvar_brca_annotated_final.csv.gz")
    df_ann["VariationID"] = df_ann["VariationID"].astype(str)
    keep = [c for c in ["VariationID","CleanRSID","GeneSymbol","label","ClinicalSignificance",
                        "Consequence","SIFT","PolyPhen","AF"] if c in df_ann.columns]
    df_meta = df_ann[keep].drop_duplicates(subset=["VariationID"])
    df_ev["VariationID"] = df_ev["VariationID"].astype(str)
    df_ev = df_ev.merge(df_meta, on="VariationID", how="left")

    OUT = ROOT / "data/processed/evidence_sentences.csv.gz"
    df_ev.to_csv(OUT, index=False, compression="gzip")
    print(f"\n✅ SAVED (precision-corrected) evidence dataset to {OUT}")

    print("\n   Sample clean evidence (one variant per row, specific token):")
    display(df_ev[["VariationID","CleanRSID","label","HGVS_Token","Sentence"]].head(6))

    # Token frequency sanity: top tokens should look like real alleles, not 'c.631'
    print("\n   Most frequent evidence tokens:")
    print(df_ev["HGVS_Token"].value_counts().head(15).to_string())
else:
    print("\n⚠️ No sentences. (Unlikely after strict fix — send me the printed token count.)")

# 9. Evidence Cleaning and Evidence-Type Tagging

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from collections import Counter

ROOT = Path("/content/evidvar")

INPUT_EV = ROOT / "data/processed/evidence_sentences.csv.gz"

if not INPUT_EV.exists():
    raise FileNotFoundError("evidence_sentences.csv.gz not found. Re-run Milestone 6B final cell.")

df = pd.read_csv(INPUT_EV)

print("Initial evidence rows:", len(df))
print("Initial distinct variants:", df["VariationID"].nunique())

# --------------------------------------------------
# 1. STRICTER TOKEN FILTER
# --------------------------------------------------

def is_clean_specific_token(tok):
    """
    Keep only tokens that look like a specific allele change.
    Reject vague tokens like c.81-, c.135-, c.442-.
    """
    if not isinstance(tok, str):
        return False

    tok = tok.lower().strip()

    if tok.startswith("c."):
        # Reject tokens ending in + or -
        if re.search(r"[+-]$", tok):
            return False

        # Accept if it contains a clear change:
        # substitution >, deletion del, duplication dup, insertion ins, nonsense *
        if re.search(r"(>|del|dup|ins|\*)", tok):
            return True

        return False

    if tok.startswith("p."):
        # Accept common protein-level change patterns
        if re.search(r"(ter|\*|fs|del|dup|ins)", tok):
            return True

        # Accept missense-like patterns: p.glu23val
        if re.search(r"p\.[a-z]{3}\d+[a-z]{3}", tok):
            return True

        return False

    return False

df["token_clean"] = df["HGVS_Token"].apply(is_clean_specific_token)

removed = df[~df["token_clean"]]
df_clean = df[df["token_clean"]].drop(columns=["token_clean"]).copy()

print("\nRemoved vague tokens:", len(removed))
print("Example removed tokens:", removed["HGVS_Token"].dropna().unique()[:10])

print("\nClean evidence rows:", len(df_clean))
print("Clean distinct variants:", df_clean["VariationID"].nunique())

# --------------------------------------------------
# 2. DEDUPLICATE
# --------------------------------------------------

df_clean = df_clean.drop_duplicates(subset=["VariationID", "PMID", "Sentence"])

print("\nAfter deduplication:", len(df_clean))

# --------------------------------------------------
# 3. COUNT HOW MANY VARIANTS EACH SENTENCE MENTIONS
# --------------------------------------------------

sent_var_counts = (
    df_clean
    .groupby(["PMID", "Sentence"])["VariationID"]
    .nunique()
    .rename("variants_in_sentence")
    .reset_index()
)

df_clean = df_clean.merge(
    sent_var_counts,
    on=["PMID", "Sentence"],
    how="left"
)

print("\nVariants-per-sentence distribution:")
print(df_clean["variants_in_sentence"].value_counts().sort_index().head(15))

# --------------------------------------------------
# 4. ADD RULE-BASED EVIDENCE TYPE FLAGS
# --------------------------------------------------

def classify_evidence_types(sentence):
    s = str(sentence).lower()

    types = []

    if any(k in s for k in [
        "assay", "functional", "minigene", "splicing assay", "rna",
        "expression", "protein", "truncat", "western", "immunoblot",
        "homologous recombination", "repair", "loss of function", "lof"
    ]):
        types.append("functional_assay")

    if any(k in s for k in [
        "segregat", "family", "pedigree", "co-segregat", "inheritance",
        "affected relatives"
    ]):
        types.append("segregation")

    if any(k in s for k in [
        "frequency", "gnomad", "1000 genome", "population",
        "allele frequency", "controls", "unaffected", "common polymorphism"
    ]):
        types.append("population_frequency")

    if any(k in s for k in [
        "case-control", "cases and controls", "patient", "association",
        "odds ratio", "risk", "cohort"
    ]):
        types.append("case_control")

    if "de novo" in s:
        types.append("de_novo")

    if any(k in s for k in [
        "in silico", "prediction", "sift", "polyphen", "cadd", "revel",
        "computational"
    ]):
        types.append("computational")

    if any(k in s for k in [
        "tumor", "cancer", "breast", "ovarian", "phenotype", "clinical"
    ]):
        types.append("clinical_context")

    if not types:
        types.append("general_text")

    return ";".join(types)

df_clean["evidence_types"] = df_clean["Sentence"].apply(classify_evidence_types)

print("\nTop evidence type combinations:")
print(df_clean["evidence_types"].value_counts().head(15))

# --------------------------------------------------
# 5. TRY TO FILL MISSING CLINICAL SIGNIFICANCE FROM INTERIM FILE
# --------------------------------------------------

meta_file = ROOT / "data/interim/brca_with_rsid.csv.gz"

if meta_file.exists():
    print("\nFilling missing metadata from brca_with_rsid.csv.gz...")
    meta = pd.read_csv(
        meta_file,
        usecols=lambda c: c in [
            "VariationID", "GeneSymbol", "ClinicalSignificance",
            "ReviewStatus", "CleanRSID"
        ]
    )
    meta["VariationID"] = meta["VariationID"].astype(str)
    meta = meta.drop_duplicates(subset=["VariationID"])

    for col in ["GeneSymbol", "ClinicalSignificance", "ReviewStatus", "CleanRSID"]:
        if col in meta.columns:
            if col in df_clean.columns:
                df_clean[col] = df_clean[col].fillna(
                    df_clean["VariationID"].map(meta.set_index("VariationID")[col])
                )
            else:
                df_clean[col] = df_clean["VariationID"].map(meta.set_index("VariationID")[col])

# --------------------------------------------------
# 6. SPLIT LABELED VS UNCERTAINTY EVIDENCE
# --------------------------------------------------

# label comes from the high-confidence supervised dataset.
# NaN means VUS/conflicting/not in supervised training pool.
labeled_mask = df_clean["label"].notna()

df_labeled = df_clean[labeled_mask].copy()
df_uncertainty = df_clean[~labeled_mask].copy()

print("\n=== FINAL CLEAN EVIDENCE SUMMARY ===")
print("Total clean evidence sentences:", len(df_clean))
print("Total clean variants with evidence:", df_clean["VariationID"].nunique())

print("\nLabeled evidence sentences:", len(df_labeled))
print("Labeled variants with evidence:", df_labeled["VariationID"].nunique())

print("\nUncertainty/VUS evidence sentences:", len(df_uncertainty))
print("Uncertainty/VUS variants with evidence:", df_uncertainty["VariationID"].nunique())

# --------------------------------------------------
# 7. SAVE OUTPUTS
# --------------------------------------------------

OUT_ALL = ROOT / "data/processed/evidence_clean_all.csv.gz"
OUT_LAB = ROOT / "data/processed/evidence_clean_labeled.csv.gz"
OUT_UNC = ROOT / "data/processed/evidence_clean_uncertainty.csv.gz"

df_clean.to_csv(OUT_ALL, index=False, compression="gzip")
df_labeled.to_csv(OUT_LAB, index=False, compression="gzip")
df_uncertainty.to_csv(OUT_UNC, index=False, compression="gzip")

print("\nSaved:")
print(OUT_ALL)
print(OUT_LAB)
print(OUT_UNC)

print("\nSample clean labeled evidence:")
display(df_labeled[
    ["VariationID", "CleanRSID", "label", "HGVS_Token",
     "variants_in_sentence", "evidence_types", "Sentence"]
].head(8))

# 10a. Optional NLP Smoke Test

In [ ]:
# =====================================================
# Milestone 7B - Test NLP Evidence Polarity on 5 Sentences
# =====================================================

import subprocess
import sys
import torch
import pandas as pd
from pathlib import Path

# 1. Install required libraries
print("Installing transformers and accelerate...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers", "accelerate", "torch"
])

# 2. Check GPU
print("\nGPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Code will run on CPU, but it may be slower.")

# 3. Load clean labeled evidence
ROOT = Path("/content/evidvar")
EVID_FILE = ROOT / "data/processed/evidence_clean_labeled.csv.gz"

if not EVID_FILE.exists():
    raise FileNotFoundError("evidence_clean_labeled.csv.gz not found. Re-run Milestone 7A.")

df = pd.read_csv(EVID_FILE)
print(f"\nLoaded {len(df)} labeled evidence sentences.")

# Take 5 diverse sample sentences
sample = df.drop_duplicates(subset=["Sentence"]).head(5).copy()

print("\nSample sentences to test:")
for i, row in sample.iterrows():
    print(f"\n--- Row {i} ---")
    print("VariationID:", row["VariationID"])
    print("CleanRSID:", row.get("CleanRSID", ""))
    print("ClinVar label:", row.get("label", ""))
    print("Evidence types:", row.get("evidence_types", ""))
    print("Variants in sentence:", row.get("variants_in_sentence", ""))
    print("Full sentence:")
    print(row["Sentence"])

# 4. Define zero-shot labels
candidate_labels = [
    "The genetic variant is pathogenic or likely pathogenic.",
    "The genetic variant is benign or likely benign.",
    "The sentence does not provide clear evidence about whether the variant is pathogenic or benign."
]

# 5. Load zero-shot classification pipeline
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1

model_candidates = [
    "MoritzLaurer/deberta-v3-base-zeroshot-v1",
    "facebook/bart-large-mnli"
]

clf = None
used_model = None

for model_name in model_candidates:
    try:
        print(f"\nTrying model: {model_name}")
        clf = pipeline(
            "zero-shot-classification",
            model=model_name,
            device=device
        )
        used_model = model_name
        print(f"Successfully loaded model: {used_model}")
        break
    except Exception as e:
        print(f"Failed to load {model_name}: {e}")

if clf is None:
    raise RuntimeError("Could not load any zero-shot classification model.")

# 6. Test on 5 sentences
print("\nRunning NLP test on 5 sentences...\n")

results = []

for idx, row in sample.iterrows():
    sentence = str(row["Sentence"])

    try:
        out = clf(
            sentence,
            candidate_labels,
            multi_label=False,
            truncation=True
        )

        top_label = out["labels"][0]
        top_score = out["scores"][0]

        # Map label to simple class
        if "pathogenic" in top_label.lower():
            polarity = "supports_pathogenic"
        elif "benign" in top_label.lower():
            polarity = "supports_benign"
        else:
            polarity = "neutral"

        results.append({
            "VariationID": row["VariationID"],
            "CleanRSID": row.get("CleanRSID", ""),
            "ClinVar_label": row.get("label", ""),
            "Evidence_types": row.get("evidence_types", ""),
            "variants_in_sentence": row.get("variants_in_sentence", ""),
            "Sentence": sentence,
            "NLP_model": used_model,
            "Raw_top_label": top_label,
            "Raw_top_score": top_score,
            "Polarity": polarity
        })

        print(f"Row {idx}")
        print("Sentence:", sentence[:180], "...")
        print("Predicted polarity:", polarity)
        print("Confidence:", round(top_score, 4))
        print("-" * 80)

    except Exception as e:
        print(f"Error on row {idx}: {e}")

df_test = pd.DataFrame(results)

TEST_OUT = ROOT / "results/tables/nlp_polarity_test_5_sentences.csv"
TEST_OUT.parent.mkdir(parents=True, exist_ok=True)
df_test.to_csv(TEST_OUT, index=False)

print("\n✅ Saved NLP test results to:")
print(TEST_OUT)

print("\nTest result table:")
display(df_test[
    ["VariationID", "CleanRSID", "ClinVar_label", "Polarity", "Raw_top_score", "Evidence_types"]
])

# 10b. Full Zero-Shot NLP Evidence Polarity

In [ ]:
# =====================================================
# Milestone 7C - Full NLP evidence polarity (resumable)
# =====================================================
import os, re, time
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from tqdm.auto import tqdm

ROOT = Path("/content/evidvar")
IN_FILE  = ROOT / "data/processed/evidence_clean_all.csv.gz"
OUT_FILE = ROOT / "data/processed/evidence_nlp_polarity.csv.gz"

if not IN_FILE.exists():
    raise FileNotFoundError("evidence_clean_all.csv.gz missing. Re-run Milestone 7A.")

df = pd.read_csv(IN_FILE)
df["VariationID"] = df["VariationID"].astype(str)
print(f"Evidence rows to annotate: {len(df):,}  (labeled={df['label'].notna().sum():,}, uncertainty={df['label'].isna().sum():,})")

# ---- circularity / provenance regex (catches 'restated database calls') ----
CIRC = re.compile(
    r"(\bclinvar\b|\boncokb\b|\bdbvar\b|\blovd\b|\bhgmd\b|\boncotator\b)"
    r"|(annotated|classified|reported|called)\s+(as\s+)?(pathogenic|benign|likely|p\s*/\s*l|b\s*/\s*l|vus|uncertain|loss.of.function|oncogenic)"
    r"|(by\s+(clinvar|oncokb|oncotator|the\s+database))"
    r"|(according to .{0,30}(guideline|clinvar|classification))",
    re.IGNORECASE,
)

# ---- load model once ----
from transformers import pipeline
device = 0 if torch.cuda.is_available() else -1
print(f"Device: {'GPU '+torch.cuda.get_device_name(0) if device==0 else 'CPU'}")
clf = pipeline("zero-shot-classification",
               model="MoritzLaurer/deberta-v3-base-zeroshot-v1",
               device=device)

LABELS = [
    "The genetic variant is pathogenic or likely pathogenic.",
    "The genetic variant is benign or likely benign.",
]

# ---- resume support ----
done_keys = set()
results = []
if OUT_FILE.exists():
    prev = pd.read_csv(OUT_FILE)
    prev["VariationID"] = prev["VariationID"].astype(str)
    results = prev.to_dict("records")
    done_keys = set(zip(prev["VariationID"], prev["PMID"].astype(str), prev["HGVS_Token"]))
    print(f"Resuming: {len(done_keys)} already done.")

pending = df[~df.apply(lambda r: (r["VariationID"], str(r["PMID"]), r["HGVS_Token"]) in done_keys, axis=1)].reset_index(drop=True)
print(f"Pending rows: {len(pending):,}")

BATCH = 16
def derive(path_s, ben_s):
    conf = float(max(path_s, ben_s))
    if path_s < 0.35 and ben_s < 0.35:
        pol = "neutral"
    elif path_s - ben_s > 0.10:
        pol = "supports_pathogenic"
    elif ben_s - path_s > 0.10:
        pol = "supports_benign"
    else:
        pol = "ambiguous"
    denom = path_s + ben_s + 1e-9
    conflict = float(1.0 - abs(path_s - ben_s) / denom) if (path_s > 0.2 and ben_s > 0.2) else 0.0
    return pol, conf, conflict

for i in tqdm(range(0, len(pending), BATCH), desc="NLP batches"):
    chunk = pending.iloc[i:i+BATCH]
    texts = chunk["Sentence"].astype(str).tolist()
    try:
        outs = clf(texts, LABELS, multi_label=True, batch_size=BATCH, truncation=True)
    except Exception as e:
        print(f"  batch {i} failed: {e}; retrying singly")
        outs = []
        for t in texts:
            try:
                outs.append(clf(t, LABELS, multi_label=True, truncation=True))
            except Exception as e2:
                outs.append({"scores": [0.0, 0.0]})
    for (_, row), o in zip(chunk.iterrows(), outs):
        sc = o["scores"]
        path_s, ben_s = float(sc[0]), float(sc[1])
        pol, conf, conflict = derive(path_s, ben_s)
        circ = bool(CIRC.search(str(row["Sentence"])))
        results.append({
            "VariationID": row["VariationID"],
            "PMID": row["PMID"],
            "HGVS_Token": row["HGVS_Token"],
            "CleanRSID": row.get("CleanRSID"),
            "label": row.get("label"),
            "evidence_types": row.get("evidence_types"),
            "variants_in_sentence": row.get("variants_in_sentence"),
            "path_score": path_s,
            "ben_score": ben_s,
            "polarity": pol,
            "confidence": conf,
            "conflict_score": conflict,
            "circular": circ,
            "Sentence": row["Sentence"],
        })
    # checkpoint every batch (cheap, 912 rows)
    pd.DataFrame(results).to_csv(OUT_FILE, index=False, compression="gzip")

df_nlp = pd.DataFrame(results)
df_nlp.to_csv(OUT_FILE, index=False, compression="gzip")
print(f"\n✅ Saved NLP polarity for {len(df_nlp):,} sentences -> {OUT_FILE}")

# ===================== QUICK PAPER-READY SUMMARY =====================
print("\n--- Polarity distribution (all) ---")
print(df_nlp["polarity"].value_counts().to_string())

print("\n--- Circularity flag ---")
print(f"Circular (restated database calls): {int(df_nlp['circular'].sum()):,} / {len(df_nlp):,}")

print("\n--- Conflict (both polarities present) ---")
print(f"Sentences with conflict_score>0.3: {int((df_nlp['conflict_score']>0.3).sum()):,}")

print("\n--- Confidence buckets ---")
print(pd.cut(df_nlp["confidence"], [0,0.4,0.6,0.8,1.0]).value_counts().sort_index().to_string())

# Agreement vs ClinVar label on the LABELED pool (abstain on neutral/ambiguous)
lab = df_nlp[df_nlp["label"].notna()].copy()
lab["pred"] = lab["polarity"].map({"supports_pathogenic":1.0,"supports_benign":0.0})
dec = lab.dropna(subset=["pred"])
acc = (dec["pred"] == dec["label"]).mean() if len(dec) else float("nan")
abstain = 1 - len(dec)/len(lab) if len(lab) else float("nan")
print("\n--- Zero-shot vs ClinVar (labeled pool) ---")
print(f"Decided sentences: {len(dec):,} / {len(lab):,}")
print(f"Abstention rate (neutral/ambiguous): {abstain:.3f}")
print(f"Agreement on decided: {acc:.3f}")

# Show the interesting cases for the error-analysis figure/table
print("\n--- Sample CIRCULAR sentences (should be down-weighted) ---")
display(df_nlp[df_nlp["circular"]][["CleanRSID","label","polarity","confidence","Sentence"]].head(3))
print("\n--- Sample CONFLICT sentences (multi-polarity) ---")
display(df_nlp[df_nlp["conflict_score"]>0.4][["CleanRSID","label","path_score","ben_score","conflict_score","Sentence"]].head(3))
print("\n--- Sample LOW-confidence pathogenic-leaning (weak evidence) ---")
display(df_nlp[(df_nlp["polarity"]=="supports_pathogenic")&(df_nlp["confidence"]<0.5)][["CleanRSID","label","confidence","Sentence"]].head(3))

# 11. Per-Variant Text Evidence Aggregation

In [ ]:
# =====================================================
# Milestone 8A - Per-variant text aggregation + diagnostics
# =====================================================
import numpy as np
import pandas as pd
import json
from pathlib import Path

ROOT = Path("/content/evidvar")
ANN   = ROOT / "data/processed/clinvar_brca_annotated_final.csv.gz"   # 16,355 labeled universe
NLP   = ROOT / "data/processed/evidence_nlp_polarity.csv.gz"          # 912 sentences

ann = pd.read_csv(ANN)
nlp = pd.read_csv(NLP)
ann["VariationID"]  = ann["VariationID"].astype(str)
nlp["VariationID"]  = nlp["VariationID"].astype(str)
nlp["PMID"]         = nlp["PMID"].astype(str)
print(f"Labeled universe: {len(ann):,} variants | NLP sentences: {len(nlp):,}")

# ---- recompute structured binary flags (same logic as Cell 5) ----
cts = ann["Consequence"].fillna("unknown").str.split(";")
ann["is_frameshift"]  = cts.apply(lambda x: int(any("frameshift" in t for t in x)))
ann["is_stop_gained"] = cts.apply(lambda x: int(any("stop_gained"  in t for t in x)))
ann["is_splice_site"] = cts.apply(lambda x: int(any(("splice_acceptor" in t or "splice_donor" in t) for t in x)))
ann["is_missense"]    = cts.apply(lambda x: int(any("missense"      in t for t in x)))
ann["is_intron"]      = cts.apply(lambda x: int(any("intron"        in t for t in x)))
ann["is_synonymous"]  = cts.apply(lambda x: int(any("synonymous"    in t for t in x)))
STRUCT = ["is_frameshift","is_stop_gained","is_splice_site","is_missense",
          "is_intron","is_synonymous","SIFT","PolyPhen","AF"]

# ---- per-sentence weight: confidence / specificity, circular excluded ----
nlp["vis"]   = pd.to_numeric(nlp["variants_in_sentence"], errors="coerce").fillna(1).clip(lower=1)
nlp["w"]     = nlp["confidence"] / nlp["vis"]
nlp.loc[nlp["circular"].astype(bool), "w"] = 0.0          # exclude restated database calls
nlp["decided"] = (~nlp["polarity"].isin(["neutral","ambiguous"])) & (~nlp["circular"].astype(bool))
nlp["wp"] = nlp["w"] * nlp["path_score"]
nlp["wb"] = nlp["w"] * nlp["ben_score"]

# ---- aggregate per variant ----
agg = (nlp.groupby("VariationID")
          .agg(text_path   = ("wp","sum"),
               text_ben    = ("wb","sum"),
               text_n      = ("decided","sum"),                 # decided non-circular sentences
               text_n_all  = ("Sentence","size"),
               text_n_circ = ("circular","sum"),
               text_conflict = ("conflict_score","max"),
               text_mean_conf= ("confidence","mean"))
          .reset_index())
agg["text_has"] = (agg["text_n"] > 0).astype(int)

# ---- merge onto labeled universe (0 where no text) ----
sup = ann.merge(agg, on="VariationID", how="left")
TEXT = ["text_path","text_ben","text_n","text_n_all","text_n_circ",
        "text_conflict","text_mean_conf","text_has"]
for c in TEXT:
    sup[c] = sup[c].fillna(0)

# ---- also keep an all-variant text table (labeled + uncertainty) for VUS analysis ----
allvar_text = agg.copy()

# ===================== DIAGNOSTICS =====================
lab_with_text = sup[sup["text_has"]==1]
print("\n--- Text coverage on labeled universe ---")
print(f"Labeled variants WITH decided text : {len(lab_with_text):,} / {len(sup):,}")
print(f"Labeled variants WITHOUT text      : {(sup['text_has']==0).sum():,}")

print("\n--- CLASS BALANCE (the artifact check) ---")
overall = sup["label"].value_counts(normalize=True).to_dict()
with_t  = lab_with_text["label"].value_counts(normalize=True).to_dict()
print(f"Overall labeled balance      : path={overall.get(1.0,0):.3f}  ben={overall.get(0.0,0):.3f}")
print(f"Text-bearing labeled balance : path={with_t.get(1.0,0):.3f}  ben={with_t.get(0.0,0):.3f}")
print(">>> If text-bearing skews pathogenic, a naive 'fusion wins' is partly an artifact;")
print("    8B therefore reports a MATCHED comparison (test-fold variants that have text).")

print("\n--- One-sidedness check (expect text_ben ~ 0) ---")
print(sup[["text_path","text_ben"]].describe().loc[["mean","50%","max"]].round(4).to_string())

print("\n--- Top 8 variants by text_path ---")
print(sup.nlargest(8,"text_path")[["VariationID","CleanRSID","label","text_path","text_ben",
                                   "text_n","text_conflict","text_n_circ"]].to_string(index=False))

print("\n--- Sample merged rows (structured + text) ---")
display(sup[["VariationID","label"]+STRUCT+TEXT].head(6))

# ---- save artifacts ----
OUT_SUP = ROOT / "data/processed/variant_features_supervised.csv.gz"
OUT_TXT = ROOT / "data/processed/variant_text_agg_all.csv.gz"
sup.to_csv(OUT_SUP, index=False, compression="gzip")
allvar_text.to_csv(OUT_TXT, index=False, compression="gzip")

diag = {
    "labeled_universe": int(len(sup)),
    "labeled_with_text": int(len(lab_with_text)),
    "overall_balance_path": float(overall.get(1.0,0)),
    "textbearing_balance_path": float(with_t.get(1.0,0)),
    "text_ben_max": float(sup["text_ben"].max()),
    "nlp_sentences": int(len(nlp)),
    "nlp_circular": int(nlp["circular"].sum()),
}
(ROOT/"results/tables").mkdir(parents=True, exist_ok=True)
with open(ROOT/"results/tables/milestone8a_diagnostics.json","w") as f:
    json.dump(diag,f,indent=2)

print(f"\nSaved:\n  {OUT_SUP}\n  {OUT_TXT}\n  {ROOT/'results/tables/milestone8a_diagnostics.json'}")

# 12. Final Bayesian Fusion, Calibration, and Robustness Analysis

In [ ]:
# =====================================================
# Milestone 8D - Honest alpha selection + prevalence-aware calibration
# =====================================================

import numpy as np
import pandas as pd
import json
import joblib
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss
)

ROOT = Path("/content/evidvar")

SUP_FILE = ROOT / "data/processed/variant_features_supervised.csv.gz"
NLP_FILE = ROOT / "data/processed/evidence_nlp_polarity.csv.gz"

assert SUP_FILE.exists(), "Missing variant_features_supervised.csv.gz. Re-run Milestone 8A."
assert NLP_FILE.exists(), "Missing evidence_nlp_polarity.csv.gz. Re-run Milestone 7C."

sup = pd.read_csv(SUP_FILE)
nlp = pd.read_csv(NLP_FILE)

sup["VariationID"] = sup["VariationID"].astype(str)
nlp["VariationID"] = nlp["VariationID"].astype(str)
nlp["PMID"] = nlp["PMID"].astype(str)

STRUCT = [
    "is_frameshift",
    "is_stop_gained",
    "is_splice_site",
    "is_missense",
    "is_intron",
    "is_synonymous",
    "SIFT",
    "PolyPhen",
    "AF"
]

RANDOM_STATE = 42
TEST_SIZE = 0.2
AF_THRESHOLD = 1e-3
CLIP = 3.0

# Safer alpha grid. We include 16 only for diagnostics, but selection will prefer <= 8.
ALPHA_GRID = [0.0, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0, 16.0]
MAX_SELECTABLE_ALPHA = 8.0

y = sup["label"].astype(float).values
idx = np.arange(len(sup))

itr, ite = train_test_split(
    idx,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

sup_tr = sup.iloc[itr].reset_index(drop=True)
sup_te = sup.iloc[ite].reset_index(drop=True)

y_tr = y[itr]
y_te = y[ite]

print(f"Train variants: {len(sup_tr):,}")
print(f"Test variants:  {len(sup_te):,}")
print(f"Train text-bearing variants: {int((sup_tr['text_n'] > 0).sum()):,}")
print(f"Test text-bearing variants:  {int((sup_te['text_n'] > 0).sum()):,}")


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def sigmoid(z):
    z = np.asarray(z, dtype=float)
    return 1.0 / (1.0 + np.exp(-z))


def logit(p, eps=1e-6):
    p = np.clip(np.asarray(p, dtype=float), eps, 1.0 - eps)
    return np.log(p / (1.0 - p))


def af_gate(af, thr=AF_THRESHOLD):
    af = np.asarray(af, dtype=float)
    gate = np.ones_like(af, dtype=float)

    valid = ~np.isnan(af)
    a = np.clip(af[valid], 0.0, None)

    gate[valid] = 1.0 / (1.0 + (a / thr) ** 2)

    return gate


def text_lr(net_text, has_text, mean_conf, gate, clip=CLIP):
    net_text = np.asarray(net_text, dtype=float)
    has_text = np.asarray(has_text, dtype=float)
    mean_conf = np.asarray(mean_conf, dtype=float)
    gate = np.asarray(gate, dtype=float)

    raw = has_text * net_text * mean_conf * gate
    return np.clip(raw, -clip, clip)


def class_weights_binary(y_true):
    """
    Balanced weights for binary labels.
    Positives get total weight 0.5, negatives get total weight 0.5.
    """
    y_true = np.asarray(y_true, dtype=float)
    pos = y_true == 1.0
    neg = y_true == 0.0

    w = np.zeros_like(y_true, dtype=float)

    if pos.sum() > 0:
        w[pos] = 0.5 / pos.sum()

    if neg.sum() > 0:
        w[neg] = 0.5 / neg.sum()

    return w


def balanced_log_loss(y_true, y_prob, eps=1e-6):
    y_true = np.asarray(y_true, dtype=float)
    y_prob = np.clip(np.asarray(y_prob, dtype=float), eps, 1.0 - eps)

    w = class_weights_binary(y_true)

    ll = -np.sum(
        w * (
            y_true * np.log(y_prob) +
            (1.0 - y_true) * np.log(1.0 - y_prob)
        )
    )

    return float(ll)


def balanced_brier(y_true, y_prob):
    y_true = np.asarray(y_true, dtype=float)
    y_prob = np.asarray(y_prob, dtype=float)

    w = class_weights_binary(y_true)

    return float(np.sum(w * (y_prob - y_true) ** 2))


def expected_calibration_error(y_true, y_prob, bins=10):
    y_true = np.asarray(y_true, dtype=float)
    y_prob = np.asarray(y_prob, dtype=float)

    if len(y_true) == 0:
        return np.nan

    edges = np.linspace(0.0, 1.0, bins + 1)
    ece = 0.0
    n = len(y_prob)

    for lo, hi in zip(edges[:-1], edges[1:]):
        if hi == 1.0:
            mask = (y_prob >= lo) & (y_prob <= hi)
        else:
            mask = (y_prob >= lo) & (y_prob < hi)

        if mask.sum() == 0:
            continue

        avg_conf = y_prob[mask].mean()
        avg_acc = y_true[mask].mean()
        ece += (mask.sum() / n) * abs(avg_conf - avg_acc)

    return float(ece)


def brier_skill_score(y_true, y_prob):
    """
    BSS relative to climatological baseline: predicting the sample prevalence.
    BSS > 0 means better than prevalence baseline.
    BSS = 0 means equal to prevalence baseline.
    BSS < 0 means worse than prevalence baseline.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_prob = np.asarray(y_prob, dtype=float)

    if len(y_true) == 0:
        return np.nan

    prevalence = float(np.mean(y_true))
    baseline_brier = brier_score_loss(y_true, np.full_like(y_prob, prevalence))

    if baseline_brier == 0:
        return np.nan

    model_brier = brier_score_loss(y_true, y_prob)

    return float(1.0 - model_brier / baseline_brier)


def calibration_slope_intercept(y_true, y_prob):
    """
    Fit logistic regression:
        logit(P(y=1)) = intercept + slope * logit(p)
    Ideal: slope = 1, intercept = 0.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_prob = np.asarray(y_prob, dtype=float)

    if len(np.unique(y_true)) < 2:
        return np.nan, np.nan

    x = logit(y_prob).reshape(-1, 1)

    try:
        lr = LogisticRegression(
            penalty=None,
            solver="lbfgs",
            max_iter=1000
        )
        lr.fit(x, y_true)

        slope = float(lr.coef_[0][0])
        intercept = float(lr.intercept_[0])

        return slope, intercept

    except Exception:
        return np.nan, np.nan


def metrics_row(name, p, yt):
    p = np.asarray(p, dtype=float)
    yt = np.asarray(yt, dtype=float)

    if len(p) == 0:
        return {
            "model": name,
            "n": 0,
            "prevalence": np.nan,
            "auroc": np.nan,
            "auprc": np.nan,
            "brier": np.nan,
            "balanced_brier": np.nan,
            "bss": np.nan,
            "ece": np.nan,
            "calib_slope": np.nan,
            "calib_intercept": np.nan
        }

    prevalence = float(np.mean(yt))

    if len(np.unique(yt)) < 2:
        auroc = np.nan
        auprc = np.nan
        slope = np.nan
        intercept = np.nan
    else:
        auroc = float(roc_auc_score(yt, p))
        auprc = float(average_precision_score(yt, p))
        slope, intercept = calibration_slope_intercept(yt, p)

    return {
        "model": name,
        "n": int(len(p)),
        "prevalence": round(prevalence, 4),
        "auroc": round(auroc, 4) if not np.isnan(auroc) else np.nan,
        "auprc": round(auprc, 4) if not np.isnan(auprc) else np.nan,
        "brier": round(float(brier_score_loss(yt, p)), 4),
        "balanced_brier": round(float(balanced_brier(yt, p)), 4),
        "bss": round(float(brier_skill_score(yt, p)), 4),
        "ece": round(float(expected_calibration_error(yt, p)), 4),
        "calib_slope": round(float(slope), 4) if not np.isnan(slope) else np.nan,
        "calib_intercept": round(float(intercept), 4) if not np.isnan(intercept) else np.nan
    }


# ------------------------------------------------------------
# Out-of-fold structural predictions for alpha selection
# ------------------------------------------------------------

print("\nFitting out-of-fold structural predictions for balanced alpha selection...")

oof_p_struct = np.full(len(sup_tr), np.nan, dtype=float)

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for fold_i, (tr_i, va_i) in enumerate(kf.split(sup_tr), start=1):
    gb_fold = HistGradientBoostingClassifier(
        max_iter=100,
        learning_rate=0.1,
        min_samples_leaf=20,
        random_state=RANDOM_STATE
    )

    gb_fold.fit(
        sup_tr.iloc[tr_i][STRUCT],
        y_tr[tr_i]
    )

    oof_p_struct[va_i] = gb_fold.predict_proba(
        sup_tr.iloc[va_i][STRUCT]
    )[:, 1]

    print(f"  Fold {fold_i}/5 done.")


tr_text_mask = (sup_tr["text_n"].values > 0)
oof_text_mask = tr_text_mask & (~np.isnan(oof_p_struct))

print(f"Out-of-fold text-bearing train variants: {int(oof_text_mask.sum()):,}")
print(f"Text-bearing train label balance: path={np.mean(y_tr[oof_text_mask] == 1):.3f}, ben={np.mean(y_tr[oof_text_mask] == 0):.3f}")


net_text_tr = sup_tr["text_path"].values - sup_tr["text_ben"].values
has_text_tr = (sup_tr["text_n"].values > 0).astype(float)
conf_tr = sup_tr["text_mean_conf"].values
gate_tr = af_gate(sup_tr["AF"].values, AF_THRESHOLD)

tl_tr = text_lr(
    net_text_tr,
    has_text_tr,
    conf_tr,
    gate_tr,
    CLIP
)


alpha_rows = []

for a in ALPHA_GRID:
    p_oof = sigmoid(
        logit(oof_p_struct[oof_text_mask]) +
        a * tl_tr[oof_text_mask]
    )

    y_sub = y_tr[oof_text_mask]

    ordinary_ll = log_loss(y_sub, p_oof, labels=[0, 1])
    bal_ll = balanced_log_loss(y_sub, p_oof)
    bal_brier = balanced_brier(y_sub, p_oof)
    ordinary_brier = brier_score_loss(y_sub, p_oof)
    bss = brier_skill_score(y_sub, p_oof)

    alpha_rows.append({
        "alpha": a,
        "selectable": int(a <= MAX_SELECTABLE_ALPHA),
        "ordinary_log_loss": round(float(ordinary_ll), 5),
        "balanced_log_loss": round(float(bal_ll), 5),
        "ordinary_brier": round(float(ordinary_brier), 5),
        "balanced_brier": round(float(bal_brier), 5),
        "bss_vs_prevalence": round(float(bss), 5)
    })

alpha_df = pd.DataFrame(alpha_rows)

print("\nAlpha sensitivity using balanced losses on OOF train text-bearing variants:")
print(alpha_df.to_string(index=False))

# Select alpha using balanced log-loss, but only among selectable alphas.
selectable = alpha_df[alpha_df["selectable"] == 1].copy()

best_bal_ll = selectable["balanced_log_loss"].min()
parsimony_threshold = best_bal_ll + 0.01

alpha_best = float(
    selectable.loc[
        selectable["balanced_log_loss"] <= parsimony_threshold,
        "alpha"
    ].min()
)

print(f"\nSelected alpha by parsimonious balanced log-loss: {alpha_best}")


# ------------------------------------------------------------
# Final structural model on full train
# ------------------------------------------------------------

print("\nTraining final structural model on full train set...")

gbm = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    min_samples_leaf=20,
    random_state=RANDOM_STATE
)

gbm.fit(sup_tr[STRUCT], y_tr)

p_struct_te = gbm.predict_proba(sup_te[STRUCT])[:, 1]
prior_te = logit(p_struct_te)


net_text_te = sup_te["text_path"].values - sup_te["text_ben"].values
has_text_te = (sup_te["text_n"].values > 0).astype(float)
conf_te = sup_te["text_mean_conf"].values
gate_te = af_gate(sup_te["AF"].values, AF_THRESHOLD)

tl_te = text_lr(
    net_text_te,
    has_text_te,
    conf_te,
    gate_te,
    CLIP
)

p_fused = sigmoid(prior_te + alpha_best * tl_te)

p_fused_nogate = sigmoid(
    prior_te +
    alpha_best *
    text_lr(
        net_text_te,
        has_text_te,
        conf_te,
        np.ones_like(gate_te),
        CLIP
    )
)


# ------------------------------------------------------------
# Main tables
# ------------------------------------------------------------

table_full = pd.DataFrame([
    metrics_row("S_structural", p_struct_te, y_te),
    metrics_row("F_fused_AFgate", p_fused, y_te),
    metrics_row("F_fused_noAFgate", p_fused_nogate, y_te)
])

te_text_idx = np.where(has_text_te.astype(bool))[0]

if len(te_text_idx) > 0:
    table_matched = pd.DataFrame([
        metrics_row("S_structural", p_struct_te[te_text_idx], y_te[te_text_idx]),
        metrics_row("F_fused_AFgate", p_fused[te_text_idx], y_te[te_text_idx]),
        metrics_row("F_fused_noAFgate", p_fused_nogate[te_text_idx], y_te[te_text_idx])
    ])
else:
    table_matched = pd.DataFrame()

print("\n=== TABLE 1: FULL TEST SET ===")
print(table_full.to_string(index=False))

print("\n=== TABLE 2: MATCHED TEXT-BEARING TEST SLICE ===")
print(table_matched.to_string(index=False))

print("\nInterpretation guide:")
print("  BSS = Brier Skill Score vs prevalence baseline.")
print("  BSS > 0 means better than simply predicting the slice prevalence.")
print("  balanced_brier penalizes the pathogenic-rich matched slice appropriately.")


# ------------------------------------------------------------
# Bootstrap absolute metrics for matched slice
# ------------------------------------------------------------

print("\nBootstrapping absolute metrics on matched text-bearing slice...")

rng = np.random.default_rng(RANDOM_STATE)

boot_rows = []

if len(te_text_idx) >= 5:
    for b in range(1000):
        samp = rng.choice(te_text_idx, size=len(te_text_idx), replace=True)

        yb = y_te[samp]
        if len(np.unique(yb)) < 2:
            continue

        ps = p_struct_te[samp]
        pf = p_fused[samp]
        pfn = p_fused_nogate[samp]

        try:
            boot_rows.append({
                "model": "S_structural",
                "auroc": roc_auc_score(yb, ps),
                "auprc": average_precision_score(yb, ps),
                "brier": brier_score_loss(yb, ps),
                "balanced_brier": balanced_brier(yb, ps),
                "bss": brier_skill_score(yb, ps),
                "ece": expected_calibration_error(yb, ps, bins=5)
            })

            boot_rows.append({
                "model": "F_fused_AFgate",
                "auroc": roc_auc_score(yb, pf),
                "auprc": average_precision_score(yb, pf),
                "brier": brier_score_loss(yb, pf),
                "balanced_brier": balanced_brier(yb, pf),
                "bss": brier_skill_score(yb, pf),
                "ece": expected_calibration_error(yb, pf, bins=5)
            })

            boot_rows.append({
                "model": "F_fused_noAFgate",
                "auroc": roc_auc_score(yb, pfn),
                "auprc": average_precision_score(yb, pfn),
                "brier": brier_score_loss(yb, pfn),
                "balanced_brier": balanced_brier(yb, pfn),
                "bss": brier_skill_score(yb, pfn),
                "ece": expected_calibration_error(yb, pfn, bins=5)
            })

        except Exception:
            continue

boot_df = pd.DataFrame(boot_rows)

if len(boot_df) > 0:
    boot_summary = (
        boot_df
        .groupby("model")
        .quantile([0.025, 0.50, 0.975])
        .unstack("model")
    )

    # Simpler readable version:
    boot_readable = []

    for model_name in ["S_structural", "F_fused_AFgate", "F_fused_noAFgate"]:
        sub = boot_df[boot_df["model"] == model_name]

        for metric in ["auroc", "auprc", "brier", "balanced_brier", "bss", "ece"]:
            q = sub[metric].quantile([0.025, 0.50, 0.975])
            boot_readable.append({
                "model": model_name,
                "metric": metric,
                "q2.5": round(float(q.loc[0.025]), 4),
                "median": round(float(q.loc[0.50]), 4),
                "q97.5": round(float(q.loc[0.975]), 4)
            })

    boot_readable_df = pd.DataFrame(boot_readable)

    print("\nBootstrap 95% CIs on matched text-bearing slice:")
    print(boot_readable_df.to_string(index=False))

    # Delta bootstrap
    delta_rows = []

    for b in range(1000):
        samp = rng.choice(te_text_idx, size=len(te_text_idx), replace=True)
        yb = y_te[samp]

        if len(np.unique(yb)) < 2:
            continue

        ps = p_struct_te[samp]
        pf = p_fused[samp]

        try:
            delta_rows.append({
                "delta_auc": roc_auc_score(yb, pf) - roc_auc_score(yb, ps),
                "delta_brier": brier_score_loss(yb, pf) - brier_score_loss(yb, ps),
                "delta_balanced_brier": balanced_brier(yb, pf) - balanced_brier(yb, ps),
                "delta_ece": expected_calibration_error(yb, pf, bins=5) - expected_calibration_error(yb, ps, bins=5),
                "delta_bss": brier_skill_score(yb, pf) - brier_skill_score(yb, ps)
            })
        except Exception:
            continue

    delta_df = pd.DataFrame(delta_rows)

    if len(delta_df) > 0:
        print("\nBootstrap deltas: fused - structural on matched slice")
        print("For Brier/ECE/balanced_brier, negative means fusion is better.")
        print("For BSS, positive means fusion is better.")
        print(delta_df.quantile([0.025, 0.50, 0.975]).T.round(4).to_string())

        print(f"\nFraction bootstrap replicates with delta_brier < 0: "
              f"{(delta_df['delta_brier'] < 0).mean():.3f}")
        print(f"Fraction bootstrap replicates with delta_balanced_brier < 0: "
              f"{(delta_df['delta_balanced_brier'] < 0).mean():.3f}")
        print(f"Fraction bootstrap replicates with delta_ece < 0: "
              f"{(delta_df['delta_ece'] < 0).mean():.3f}")
        print(f"Fraction bootstrap replicates with delta_bss > 0: "
              f"{(delta_df['delta_bss'] > 0).mean():.3f}")
else:
    boot_readable_df = pd.DataFrame()
    delta_df = pd.DataFrame()
    print("Not enough matched text-bearing test variants for bootstrap.")


# ------------------------------------------------------------
# Alpha sensitivity on test, descriptive only
# ------------------------------------------------------------

alpha_sens_rows = []

for a in ALPHA_GRID:
    p_a = sigmoid(prior_te + a * tl_te)

    alpha_sens_rows.append(metrics_row(f"alpha={a}_full", p_a, y_te))

    if len(te_text_idx) > 0:
        alpha_sens_rows.append(
            metrics_row(
                f"alpha={a}_matched",
                p_a[te_text_idx],
                y_te[te_text_idx]
            )
        )

alpha_sens_df = pd.DataFrame(alpha_sens_rows)

print("\n=== ALPHA SENSITIVITY, DESCRIPTIVE TEST SLICES ===")
print(alpha_sens_df.to_string(index=False))


# ------------------------------------------------------------
# AF threshold sensitivity on matched slice
# ------------------------------------------------------------

af_thr_rows = []

af_thresholds = [0.0002, 0.0005, 0.001, 0.005, 0.01]

for thr in af_thresholds:
    gate_thr = af_gate(sup_te["AF"].values, thr)

    tl_thr = text_lr(
        net_text_te,
        has_text_te,
        conf_te,
        gate_thr,
        CLIP
    )

    p_thr = sigmoid(prior_te + alpha_best * tl_thr)

    if len(te_text_idx) > 0:
        af_thr_rows.append(
            metrics_row(
                f"AFthr={thr}_matched",
                p_thr[te_text_idx],
                y_te[te_text_idx]
            )
        )

af_thr_df = pd.DataFrame(af_thr_rows)

print("\n=== AF GATE SENSITIVITY ON MATCHED SLICE ===")
print(af_thr_df.to_string(index=False))


# ------------------------------------------------------------
# rs11571833 worked example
# ------------------------------------------------------------

rs_mask = sup_te["CleanRSID"].astype(str) == "rs11571833"

if rs_mask.any():
    loc = np.where(rs_mask.values)[0][0]

    print("\n=== WORKED EXAMPLE: rs11571833 ===")
    print(f"  ClinVar label       : {sup_te['label'].iloc[loc]}")
    print(f"  AF                  : {sup_te['AF'].iloc[loc]}")
    print(f"  net_text path-ben   : {net_text_te[loc]:.3f}")
    print(f"  decided sentences   : {sup_te['text_n'].iloc[loc]:.0f}")
    print(f"  AF-gate multiplier  : {gate_te[loc]:.3f}")
    print(f"  text_lr contribution: {tl_te[loc]:.3f}")
    print(f"  selected alpha      : {alpha_best}")
    print(f"  P(structural only)  : {p_struct_te[loc]:.4f}")
    print(f"  P(fused no gate)    : {p_fused_nogate[loc]:.4f}")
    print(f"  P(fused + AF gate)  : {p_fused[loc]:.4f}")
else:
    print("\nrs11571833 is not in this test fold. It may be in train fold.")


# ------------------------------------------------------------
# Figures
# ------------------------------------------------------------

TABLES = ROOT / "results" / "tables"
FIGS = ROOT / "results" / "figures"
MODELS = ROOT / "results" / "models"

TABLES.mkdir(parents=True, exist_ok=True)
FIGS.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)


def reliability_curve(ax, yt, p, label):
    edges = np.linspace(0, 1, 11)
    xs, ys, ns = [], [], []

    for lo, hi in zip(edges[:-1], edges[1:]):
        if hi == 1.0:
            mask = (p >= lo) & (p <= hi)
        else:
            mask = (p >= lo) & (p < hi)

        if mask.sum() == 0:
            continue

        xs.append(p[mask].mean())
        ys.append(yt[mask].mean())
        ns.append(mask.sum())

    ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
    ax.plot(xs, ys, "o-", label=label)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("predicted P(pathogenic)")
    ax.set_ylabel("observed frequency")
    ax.set_title(label)
    ax.legend(fontsize=8)


try:
    # Full test calibration
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.5))

    reliability_curve(axs[0], y_te, p_struct_te, "S structural")
    reliability_curve(axs[1], y_te, p_fused, "F fused + AF gate")
    reliability_curve(axs[2], y_te, p_fused_nogate, "F fused no AF gate")

    fig.suptitle("Reliability diagrams: full test set")
    fig.tight_layout()
    fig.savefig(FIGS / "fig_calibration_full_8D.png", dpi=180)
    plt.close(fig)

    # Matched slice calibration
    if len(te_text_idx) > 0:
        fig, axs = plt.subplots(1, 3, figsize=(15, 4.5))

        reliability_curve(axs[0], y_te[te_text_idx], p_struct_te[te_text_idx], "S structural")
        reliability_curve(axs[1], y_te[te_text_idx], p_fused[te_text_idx], "F fused + AF gate")
        reliability_curve(axs[2], y_te[te_text_idx], p_fused_nogate[te_text_idx], "F fused no AF gate")

        fig.suptitle("Reliability diagrams: matched text-bearing slice")
        fig.tight_layout()
        fig.savefig(FIGS / "fig_calibration_matched_8D.png", dpi=180)
        plt.close(fig)

    # Alpha sensitivity plot
    fig, ax = plt.subplots(figsize=(7, 4.5))

    full_sens = alpha_sens_df[alpha_sens_df["model"].str.endswith("_full")].copy()
    full_sens["alpha"] = full_sens["model"].str.extract(r"alpha=([0-9.]+)_full").astype(float)

    matched_sens = alpha_sens_df[alpha_sens_df["model"].str.endswith("_matched")].copy()
    matched_sens["alpha"] = matched_sens["model"].str.extract(r"alpha=([0-9.]+)_matched").astype(float)

    ax.plot(full_sens["alpha"], full_sens["auroc"], "o-", label="Full AUROC")
    ax.plot(full_sens["alpha"], full_sens["brier"], "s-", label="Full Brier")

    if len(matched_sens) > 0:
        ax.plot(matched_sens["alpha"], matched_sens["auroc"], "o--", label="Matched AUROC")
        ax.plot(matched_sens["alpha"], matched_sens["balanced_brier"], "s--", label="Matched balanced Brier")

    ax.set_xscale("symlog", linthresh=0.5)
    ax.set_xlabel("alpha")
    ax.set_title("Alpha sensitivity")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(FIGS / "fig_alpha_sensitivity_8D.png", dpi=180)
    plt.close(fig)

    print("\nFigures saved.")

except Exception as e:
    print(f"Plotting skipped: {e}")


# ------------------------------------------------------------
# Save artifacts
# ------------------------------------------------------------

table_full.to_csv(TABLES / "table1_full_test_8D.csv", index=False)
table_matched.to_csv(TABLES / "table2_matched_test_8D.csv", index=False)
alpha_df.to_csv(TABLES / "alpha_selection_balanced_oof_8D.csv", index=False)
alpha_sens_df.to_csv(TABLES / "alpha_sensitivity_test_8D.csv", index=False)
af_thr_df.to_csv(TABLES / "af_threshold_sensitivity_matched_8D.csv", index=False)

if len(boot_readable_df) > 0:
    boot_readable_df.to_csv(TABLES / "bootstrap_matched_absolute_8D.csv", index=False)

if len(delta_df) > 0:
    delta_df.to_csv(TABLES / "bootstrap_matched_deltas_8D.csv", index=False)

joblib.dump(gbm, MODELS / "structural_model_8D.pkl")

fusion_config = {
    "alpha": float(alpha_best),
    "alpha_grid": ALPHA_GRID,
    "max_selectable_alpha": MAX_SELECTABLE_ALPHA,
    "alpha_selection_metric": "balanced_log_loss_oof_train_text_bearing",
    "af_threshold": float(AF_THRESHOLD),
    "clip": float(CLIP),
    "random_state": int(RANDOM_STATE),
    "test_size": float(TEST_SIZE),
    "struct_features": STRUCT,
    "n_train": int(len(sup_tr)),
    "n_test": int(len(sup_te)),
    "n_train_text_bearing": int((sup_tr["text_n"] > 0).sum()),
    "n_test_text_bearing": int((sup_te["text_n"] > 0).sum())
}

with open(MODELS / "fusion_config_8D.json", "w") as f:
    json.dump(fusion_config, f, indent=2)

print("\nSaved artifacts:")
print(f"  {TABLES / 'table1_full_test_8D.csv'}")
print(f"  {TABLES / 'table2_matched_test_8D.csv'}")
print(f"  {TABLES / 'alpha_selection_balanced_oof_8D.csv'}")
print(f"  {MODELS / 'structural_model_8D.pkl'}")
print(f"  {MODELS / 'fusion_config_8D.json'}")

print("\n✅ Milestone 8D complete.")

# 13. Supplementary Case Studies and Feature Importance

In [ ]:
# =====================================================
# Milestone 8C - Robustness, bootstrap CIs, case studies, final artifacts
# =====================================================

import numpy as np
import pandas as pd
import json
import joblib
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss
)
from sklearn.inspection import permutation_importance

ROOT = Path("/content/evidvar")

SUP_FILE = ROOT / "data/processed/variant_features_supervised.csv.gz"
NLP_FILE = ROOT / "data/processed/evidence_nlp_polarity.csv.gz"

assert SUP_FILE.exists(), "Missing variant_features_supervised.csv.gz. Re-run Milestone 8A."
assert NLP_FILE.exists(), "Missing evidence_nlp_polarity.csv.gz. Re-run Milestone 7C."

sup = pd.read_csv(SUP_FILE)
nlp = pd.read_csv(NLP_FILE)

sup["VariationID"] = sup["VariationID"].astype(str)
nlp["VariationID"] = nlp["VariationID"].astype(str)
nlp["PMID"] = nlp["PMID"].astype(str)

STRUCT = [
    "is_frameshift",
    "is_stop_gained",
    "is_splice_site",
    "is_missense",
    "is_intron",
    "is_synonymous",
    "SIFT",
    "PolyPhen",
    "AF"
]

RANDOM_STATE = 42
TEST_SIZE = 0.2
AF_THRESHOLD = 1e-3
CLIP = 3.0

y = sup["label"].astype(float).values
idx = np.arange(len(sup))

itr, ite = train_test_split(
    idx,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

sup_tr = sup.iloc[itr].reset_index(drop=True)
sup_te = sup.iloc[ite].reset_index(drop=True)

y_tr = y[itr]
y_te = y[ite]

print(f"Train variants: {len(sup_tr):,}")
print(f"Test variants:  {len(sup_te):,}")
print(f"Train text-bearing variants: {int((sup_tr['text_n'] > 0).sum()):,}")
print(f"Test text-bearing variants:  {int((sup_te['text_n'] > 0).sum()):,}")


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def sigmoid(z):
    z = np.asarray(z, dtype=float)
    return 1.0 / (1.0 + np.exp(-z))


def logit(p, eps=1e-6):
    p = np.clip(np.asarray(p, dtype=float), eps, 1.0 - eps)
    return np.log(p / (1.0 - p))


def af_gate(af, thr=AF_THRESHOLD):
    """
    ACMG-inspired population-frequency gate.
    High allele frequency suppresses text evidence.
    Missing AF is treated as unknown and receives gate=1.
    """
    af = np.asarray(af, dtype=float)
    gate = np.ones_like(af, dtype=float)

    valid = ~np.isnan(af)
    a = np.clip(af[valid], 0.0, None)

    gate[valid] = 1.0 / (1.0 + (a / thr) ** 2)

    return gate


def text_lr(net_text, has_text, mean_conf, gate, clip=CLIP):
    """
    Text log-likelihood-ratio channel.

    Rules:
    - If no decided non-circular text, contribution is exactly zero.
    - Contribution is scaled by mean confidence.
    - Contribution is suppressed by allele-frequency gate.
    - Contribution is clipped to avoid one sentence dominating.
    """
    net_text = np.asarray(net_text, dtype=float)
    has_text = np.asarray(has_text, dtype=float)
    mean_conf = np.asarray(mean_conf, dtype=float)
    gate = np.asarray(gate, dtype=float)

    raw = has_text * net_text * mean_conf * gate
    return np.clip(raw, -clip, clip)


def expected_calibration_error(y_true, y_prob, bins=10):
    y_true = np.asarray(y_true, dtype=float)
    y_prob = np.asarray(y_prob, dtype=float)

    if len(y_true) == 0:
        return np.nan

    edges = np.linspace(0.0, 1.0, bins + 1)
    ece = 0.0
    n = len(y_prob)

    for lo, hi in zip(edges[:-1], edges[1:]):
        if hi == 1.0:
            mask = (y_prob >= lo) & (y_prob <= hi)
        else:
            mask = (y_prob >= lo) & (y_prob < hi)

        if mask.sum() == 0:
            continue

        avg_conf = y_prob[mask].mean()
        avg_acc = y_true[mask].mean()
        ece += (mask.sum() / n) * abs(avg_conf - avg_acc)

    return float(ece)


def metrics_row(name, p, yt):
    p = np.asarray(p, dtype=float)
    yt = np.asarray(yt, dtype=float)

    if len(p) == 0:
        return {
            "model": name,
            "n": 0,
            "auroc": np.nan,
            "auprc": np.nan,
            "brier": np.nan,
            "ece": np.nan
        }

    if len(np.unique(yt)) < 2:
        auroc = np.nan
        auprc = np.nan
    else:
        auroc = roc_auc_score(yt, p)
        auprc = average_precision_score(yt, p)

    return {
        "model": name,
        "n": int(len(p)),
        "auroc": round(float(auroc), 4) if not np.isnan(auroc) else np.nan,
        "auprc": round(float(auprc), 4) if not np.isnan(auprc) else np.nan,
        "brier": round(float(brier_score_loss(yt, p)), 4),
        "ece": round(float(expected_calibration_error(yt, p)), 4)
    }


# ------------------------------------------------------------
# Out-of-fold structural predictions for alpha selection
# ------------------------------------------------------------

print("\nFitting out-of-fold structural predictions for alpha selection...")

oof_p_struct = np.full(len(sup_tr), np.nan, dtype=float)

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for fold_i, (tr_i, va_i) in enumerate(kf.split(sup_tr), start=1):
    gb_fold = HistGradientBoostingClassifier(
        max_iter=100,
        learning_rate=0.1,
        min_samples_leaf=20,
        random_state=RANDOM_STATE
    )

    gb_fold.fit(
        sup_tr.iloc[tr_i][STRUCT],
        y_tr[tr_i]
    )

    oof_p_struct[va_i] = gb_fold.predict_proba(
        sup_tr.iloc[va_i][STRUCT]
    )[:, 1]

    print(f"  Fold {fold_i}/5 done.")


tr_text_mask = (sup_tr["text_n"].values > 0)
oof_text_mask = tr_text_mask & (~np.isnan(oof_p_struct))

print(f"Out-of-fold text-bearing train variants: {int(oof_text_mask.sum()):,}")


net_text_tr = sup_tr["text_path"].values - sup_tr["text_ben"].values
has_text_tr = (sup_tr["text_n"].values > 0).astype(float)
conf_tr = sup_tr["text_mean_conf"].values
gate_tr = af_gate(sup_tr["AF"].values, AF_THRESHOLD)

tl_tr = text_lr(
    net_text_tr,
    has_text_tr,
    conf_tr,
    gate_tr,
    CLIP
)


alphas = [0.0, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0, 16.0]

alpha_rows = []

for a in alphas:
    p_oof = sigmoid(
        logit(oof_p_struct[oof_text_mask]) +
        a * tl_tr[oof_text_mask]
    )

    ll = log_loss(
        y_tr[oof_text_mask],
        p_oof,
        labels=[0, 1]
    )

    br = brier_score_loss(
        y_tr[oof_text_mask],
        p_oof
    )

    alpha_rows.append({
        "alpha": a,
        "oof_log_loss": round(float(ll), 5),
        "oof_brier": round(float(br), 5)
    })

alpha_df = pd.DataFrame(alpha_rows)

min_ll = alpha_df["oof_log_loss"].min()
parsimony_threshold = min_ll + 0.01

alpha_best = float(
    alpha_df.loc[
        alpha_df["oof_log_loss"] <= parsimony_threshold,
        "alpha"
    ].min()
)

print("\nAlpha sensitivity on out-of-fold train text-bearing variants:")
print(alpha_df.to_string(index=False))
print(f"\nSelected alpha by parsimonious OOF log-loss: {alpha_best}")


# ------------------------------------------------------------
# Final structural model on full train
# ------------------------------------------------------------

print("\nTraining final structural model on full train set...")

gbm = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    min_samples_leaf=20,
    random_state=RANDOM_STATE
)

gbm.fit(sup_tr[STRUCT], y_tr)

p_struct_te = gbm.predict_proba(sup_te[STRUCT])[:, 1]
prior_te = logit(p_struct_te)


net_text_te = sup_te["text_path"].values - sup_te["text_ben"].values
has_text_te = (sup_te["text_n"].values > 0).astype(float)
conf_te = sup_te["text_mean_conf"].values
gate_te = af_gate(sup_te["AF"].values, AF_THRESHOLD)

tl_te = text_lr(
    net_text_te,
    has_text_te,
    conf_te,
    gate_te,
    CLIP
)

p_fused = sigmoid(prior_te + alpha_best * tl_te)

p_fused_nogate = sigmoid(
    prior_te +
    alpha_best *
    text_lr(
        net_text_te,
        has_text_te,
        conf_te,
        np.ones_like(gate_te),
        CLIP
    )
)


# ------------------------------------------------------------
# Main tables
# ------------------------------------------------------------

table_full = pd.DataFrame([
    metrics_row("S_structural", p_struct_te, y_te),
    metrics_row("F_fused_AFgate", p_fused, y_te),
    metrics_row("F_fused_noAFgate", p_fused_nogate, y_te)
])

te_text_idx = np.where(has_text_te.astype(bool))[0]

if len(te_text_idx) > 0:
    table_matched = pd.DataFrame([
        metrics_row("S_structural", p_struct_te[te_text_idx], y_te[te_text_idx]),
        metrics_row("F_fused_AFgate", p_fused[te_text_idx], y_te[te_text_idx]),
        metrics_row("F_fused_noAFgate", p_fused_nogate[te_text_idx], y_te[te_text_idx])
    ])
else:
    table_matched = pd.DataFrame()

print("\n=== TABLE 1: FULL TEST SET ===")
print(table_full.to_string(index=False))

print("\n=== TABLE 2: MATCHED TEXT-BEARING TEST SLICE ===")
print(table_matched.to_string(index=False))


# ------------------------------------------------------------
# Bootstrap CI for matched slice deltas
# ------------------------------------------------------------

print("\nBootstrapping matched text-bearing slice...")

rng = np.random.default_rng(RANDOM_STATE)

boot_rows = []

if len(te_text_idx) >= 5:
    for b in range(1000):
        samp = rng.choice(te_text_idx, size=len(te_text_idx), replace=True)

        yb = y_te[samp]
        if len(np.unique(yb)) < 2:
            continue

        ps = p_struct_te[samp]
        pf = p_fused[samp]

        try:
            auc_s = roc_auc_score(yb, ps)
            auc_f = roc_auc_score(yb, pf)
        except Exception:
            continue

        br_s = brier_score_loss(yb, ps)
        br_f = brier_score_loss(yb, pf)

        ece_s = expected_calibration_error(yb, ps, bins=5)
        ece_f = expected_calibration_error(yb, pf, bins=5)

        boot_rows.append({
            "delta_auc": auc_f - auc_s,
            "delta_brier": br_f - br_s,
            "delta_ece": ece_f - ece_s
        })

boot_df = pd.DataFrame(boot_rows)

if len(boot_df) > 0:
    boot_ci = boot_df.quantile([0.025, 0.50, 0.975]).T
    boot_ci.columns = ["q2.5", "median", "q97.5"]

    print("\nBootstrap deltas: fused - structural on matched slice")
    print("Negative delta_brier and delta_ece mean fusion is better calibrated.")
    print(boot_ci.round(4).to_string())

    print(f"\nFraction bootstrap replicates with delta_brier < 0: "
          f"{(boot_df['delta_brier'] < 0).mean():.3f}")
    print(f"Fraction bootstrap replicates with delta_ece < 0: "
          f"{(boot_df['delta_ece'] < 0).mean():.3f}")
else:
    boot_ci = pd.DataFrame()
    print("Not enough matched text-bearing test variants for bootstrap.")


# ------------------------------------------------------------
# Alpha sensitivity on test, descriptive only
# ------------------------------------------------------------

alpha_sens_rows = []

for a in alphas:
    p_a = sigmoid(prior_te + a * tl_te)

    alpha_sens_rows.append(metrics_row(f"alpha={a}_full", p_a, y_te))

    if len(te_text_idx) > 0:
        alpha_sens_rows.append(
            metrics_row(
                f"alpha={a}_matched",
                p_a[te_text_idx],
                y_te[te_text_idx]
            )
        )

alpha_sens_df = pd.DataFrame(alpha_sens_rows)

print("\n=== ALPHA SENSITIVITY, DESCRIPTIVE TEST SLICES ===")
print(alpha_sens_df.to_string(index=False))


# ------------------------------------------------------------
# AF threshold sensitivity on matched slice
# ------------------------------------------------------------

af_thr_rows = []

af_thresholds = [0.0002, 0.0005, 0.001, 0.005, 0.01]

for thr in af_thresholds:
    gate_thr = af_gate(sup_te["AF"].values, thr)

    tl_thr = text_lr(
        net_text_te,
        has_text_te,
        conf_te,
        gate_thr,
        CLIP
    )

    p_thr = sigmoid(prior_te + alpha_best * tl_thr)

    if len(te_text_idx) > 0:
        af_thr_rows.append(
            metrics_row(
                f"AFthr={thr}_matched",
                p_thr[te_text_idx],
                y_te[te_text_idx]
            )
        )

af_thr_df = pd.DataFrame(af_thr_rows)

print("\n=== AF GATE SENSITIVITY ON MATCHED SLICE ===")
print(af_thr_df.to_string(index=False))


# ------------------------------------------------------------
# rs11571833 worked example
# ------------------------------------------------------------

rs_mask = sup_te["CleanRSID"].astype(str) == "rs11571833"

if rs_mask.any():
    loc = np.where(rs_mask.values)[0][0]

    print("\n=== WORKED EXAMPLE: rs11571833 ===")
    print(f"  ClinVar label       : {sup_te['label'].iloc[loc]}")
    print(f"  AF                  : {sup_te['AF'].iloc[loc]}")
    print(f"  net_text path-ben   : {net_text_te[loc]:.3f}")
    print(f"  decided sentences   : {sup_te['text_n'].iloc[loc]:.0f}")
    print(f"  AF-gate multiplier  : {gate_te[loc]:.3f}")
    print(f"  text_lr contribution: {tl_te[loc]:.3f}")
    print(f"  P(structural only)  : {p_struct_te[loc]:.4f}")
    print(f"  P(fused no gate)    : {p_fused_nogate[loc]:.4f}")
    print(f"  P(fused + AF gate)  : {p_fused[loc]:.4f}")
else:
    print("\nrs11571833 is not in this test fold. It may be in train fold.")


# ------------------------------------------------------------
# Case studies: top text-bearing test variants
# ------------------------------------------------------------

nlp_dec = nlp[
    (~nlp["polarity"].isin(["neutral", "ambiguous"])) &
    (~nlp["circular"].astype(bool))
].copy()

example_sent = (
    nlp_dec
    .sort_values("confidence", ascending=False)
    .groupby("VariationID")
    .first()
    .reset_index()
)

example_sent = example_sent[[
    "VariationID",
    "PMID",
    "Sentence",
    "confidence",
    "evidence_types",
    "path_score",
    "ben_score"
]]

cases = sup_te[sup_te["text_n"] > 0].copy()

cases["p_struct"] = p_struct_te[cases.index]
cases["p_fused"] = p_fused[cases.index]
cases["p_fused_nogate"] = p_fused_nogate[cases.index]
cases["af_gate"] = gate_te[cases.index]
cases["text_lr"] = tl_te[cases.index]

cases = cases.merge(example_sent, on="VariationID", how="left")

case_cols = [
    "VariationID",
    "CleanRSID",
    "label",
    "ClinicalSignificance",
    "Consequence",
    "AF",
    "text_path",
    "text_ben",
    "text_n",
    "text_conflict",
    "text_n_circ",
    "p_struct",
    "p_fused_nogate",
    "p_fused",
    "af_gate",
    "text_lr",
    "PMID",
    "confidence",
    "evidence_types",
    "Sentence"
]

case_cols = [c for c in case_cols if c in cases.columns]

cases_out = cases[case_cols].sort_values(
    ["text_path", "confidence"],
    ascending=False
).head(25)

print("\n=== TOP CASE STUDIES: TEXT-BEARING TEST VARIANTS ===")
display(
    cases_out[[
        "CleanRSID",
        "label",
        "AF",
        "text_path",
        "text_n",
        "p_struct",
        "p_fused_nogate",
        "p_fused",
        "af_gate"
    ]].head(10)
)


# ------------------------------------------------------------
# Structural feature importance
# ------------------------------------------------------------

print("\nComputing permutation importance for structural model...")

try:
    pi = permutation_importance(
        gbm,
        sup_te[STRUCT],
        y_te,
        n_repeats=5,
        random_state=RANDOM_STATE,
        scoring="roc_auc",
        n_jobs=-1
    )

    imp_df = pd.DataFrame({
        "feature": STRUCT,
        "mean_auc_drop": pi.importances_mean,
        "std_auc_drop": pi.importances_std
    }).sort_values("mean_auc_drop", ascending=False)

    print("\nTop structural features by permutation importance:")
    print(imp_df.round(4).to_string(index=False))

except Exception as e:
    imp_df = pd.DataFrame()
    print(f"Permutation importance skipped: {e}")


# ------------------------------------------------------------
# Save artifacts
# ------------------------------------------------------------

TABLES = ROOT / "results" / "tables"
FIGS = ROOT / "results" / "figures"
MODELS = ROOT / "results" / "models"

TABLES.mkdir(parents=True, exist_ok=True)
FIGS.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

table_full.to_csv(TABLES / "table1_full_test_8C.csv", index=False)
table_matched.to_csv(TABLES / "table2_matched_test_8C.csv", index=False)
alpha_df.to_csv(TABLES / "alpha_selection_oof_8C.csv", index=False)
alpha_sens_df.to_csv(TABLES / "alpha_sensitivity_test_8C.csv", index=False)
af_thr_df.to_csv(TABLES / "af_threshold_sensitivity_matched_8C.csv", index=False)
cases_out.to_csv(TABLES / "case_studies_text_bearing_test_8C.csv.gz", index=False, compression="gzip")

if len(boot_df) > 0:
    boot_df.to_csv(TABLES / "bootstrap_matched_deltas_8C.csv", index=False)
    boot_ci.to_csv(TABLES / "bootstrap_matched_ci_8C.csv")

if len(imp_df) > 0:
    imp_df.to_csv(TABLES / "structural_feature_importance_8C.csv", index=False)

joblib.dump(gbm, MODELS / "structural_model_8C.pkl")

fusion_config = {
    "alpha": float(alpha_best),
    "af_threshold": float(AF_THRESHOLD),
    "clip": float(CLIP),
    "random_state": int(RANDOM_STATE),
    "test_size": float(TEST_SIZE),
    "struct_features": STRUCT,
    "n_train": int(len(sup_tr)),
    "n_test": int(len(sup_te)),
    "n_train_text_bearing": int((sup_tr["text_n"] > 0).sum()),
    "n_test_text_bearing": int((sup_te["text_n"] > 0).sum())
}

with open(MODELS / "fusion_config_8C.json", "w") as f:
    json.dump(fusion_config, f, indent=2)

print("\nSaved artifacts:")
print(f"  {TABLES / 'table1_full_test_8C.csv'}")
print(f"  {TABLES / 'table2_matched_test_8C.csv'}")
print(f"  {TABLES / 'case_studies_text_bearing_test_8C.csv.gz'}")
print(f"  {MODELS / 'structural_model_8C.pkl'}")
print(f"  {MODELS / 'fusion_config_8C.json'}")

print("\n✅ Milestone 8C complete.")

# 14. Generate Repository Documentation

In [ ]:
# =====================================================
# Milestone 9A - Write repository documents (README + docs/)
# =====================================================
from pathlib import Path

ROOT = Path("/content/evidvar")
DOCS = ROOT / "docs"
DOCS.mkdir(parents=True, exist_ok=True)

# ---------------- README.md ----------------
(ROOT / "README.md").write_text(r"""# EvidVar

**EvidVar** is a reproducible NLP + probabilistic‑genetics pipeline that interprets
BRCA1/BRCA2 germline variants by fusing structured genomic annotations (ClinVar,
Ensembl) with sentence‑level evidence mined from biomedical literature (PubMed),
and by quantifying *uncertainty* rather than only predicting a label.

## Research question

> Can textual evidence extracted from biomedical literature improve **calibrated**
> probabilistic classification of BRCA1/BRCA2 variants beyond structured genomic
> annotations alone — and can that evidence be integrated *safely* (gated by
> confidence, specificity, provenance and population frequency) so that sparse,
> one‑sided literature never damages a strong structural baseline?

## Headline result (honest framing)

Literature evidence is **sparse** (usable decided, non‑circular sentences exist for
~0.9% of high‑confidence variants: 149/16,355) and **one‑sided** (the zero‑shot
extractor yields essentially no benign‑supporting evidence). EvidVar therefore does
**not** claim a large discrimination gain. It claims a *reproducible, evidence‑gated,
calibration‑focused* fusion that:

1. leaves variants without usable text **unchanged by construction**;
2. on the text‑bearing subset **consistently improves calibration** (Brier, balanced
   Brier, ECE and Brier Skill Score all favoured in **100% of 1,000 bootstrap
   replicates**; AUROC unchanged);
3. corrects residual one‑sided false upgrades on common benign polymorphisms via an
   **ACMG‑inspired (BA1/BS1) population‑frequency gate** (worked example: rs11571833);
4. releases a **912‑sentence annotated evidence benchmark** for variant interpretation.

Full‑set performance: structural AUROC **0.903** (BSS **0.51**); fused AUROC **0.904**.
See `docs/results_summary.md` for every table and figure.

## Pipeline
ClinVar variant_summary ──
├─► normalize + label + quality filter ─► train/val/test splits
Ensembl VEP (rsID) ─────── │
▼
structured features (consequence flags, SIFT, PolyPhen, AF)
│
PubMed BRCA abstracts ─► HGVS sentence mining ─► zero‑shot polarity (DeBERTa‑v3)
│ + confidence + conflict + circularity
▼
per‑variant text aggregation (confidence/specificity‑weighted)
│
Bayesian log‑odds fusion = prior(structural)
+ α · gated_text_LR
▼
calibrated P(pathogenic) + uncertainty + evidence explanation


## Data sources (all public)

- **ClinVar** `variant_summary.txt.gz` (NCBI FTP) — labels, review status, HGVS (`Name`), rsID (`RS# (dbSNP)`).
- **Ensembl REST** `/vep/human/id/` and `/variation/human/` — consequence, SIFT, PolyPhen, allele frequency.
- **PubMed / NCBI E‑utilities** — BRCA1/2 abstracts (esearch + efetch).
- **Zero‑shot NLI model**: `MoritzLaurer/deberta-v3-base-zeroshot-v1` (Hugging Face).

## Reproduce (Colab, in order)

The project was built as an ordered sequence of notebook cells; run them top‑to‑bottom:

| Cell | Purpose | Output |
|------|---------|--------|
| 1 | env + folders | `data/`, `src/`, `results/` skeleton |
| 2 | download ClinVar, extract BRCA + rsID | `data/interim/brca_with_rsid.csv.gz` |
| 3 | label map + quality filter + stratified split | `data/processed/{train,val,test}_set.csv.gz` |
| 4 | Ensembl VEP annotation (batched, cached, resumable) | `data/processed/clinvar_brca_annotated_final.csv.gz` |
| 5 | structural baseline (HistGradientBoosting) | `models/baseline_structural_model.pkl` |
| 6A | PubMed abstract fetch (5,000 → ~4,645) | `data/interim/pubmed_abstracts.csv.gz` |
| 6B | strict HGVS sentence mining (chunked, memory‑safe) | `data/processed/evidence_sentences.csv.gz` |
| 7A | token/dedup cleanup + evidence‑type flags | `data/processed/evidence_clean_*.csv.gz` |
| 7B/7C | zero‑shot polarity + confidence + conflict + circularity | `data/processed/evidence_nlp_polarity.csv.gz` |
| 8A | per‑variant text aggregation + balance diagnostics | `data/processed/variant_features_supervised.csv.gz` |
| 8B/8C | fusion + bootstrap + ablation + case studies | `results/tables/*_8C.csv`, figures |
| 8D | **balanced** α selection + prevalence‑aware calibration | `results/tables/*_8D.csv`, `models/fusion_config_8D.json` |
| 9A | this documentation | `README.md`, `docs/*.md` |

Raw ClinVar (~435 MB) is **not** committed; `src/` scripts re‑download it. Processed
tables, models, figures and the benchmark **are** committed (small).

## Released benchmark

`data/processed/evidence_nlp_polarity.csv.gz` — 912 BRCA1/2 evidence sentences with
variant ID, PMID, matched HGVS token, evidence type, zero‑shot polarity, confidence,
conflict score and circularity flag. Intended as a starting point for evaluating
literature‑based variant evidence extraction. See `docs/methods.md` §Evidence mining.

## Limitations (read before citing)

See `docs/limitations.md`. In brief: ~0.9% text coverage; one‑sided text signal;
the text‑bearing subset is 90% pathogenic and n=30 (so absolute BSS there is negative
for *all* models — a subset artefact, explained in `results_summary.md`); ClinVar
labels are curated consensus, not gold truth; abstracts (not full text) were mined;
the NLP step is zero‑shot, not fine‑tuned; no temporal split yet. **Not for clinical use.**

## License & citation

Code: MIT (see `LICENSE`). Data: respect NCBI/Ensembl terms of use; cite ClinVar,
Ensembl and PubMed/NCBI as per their policies. If you use the benchmark, cite this
repository and the underlying resources.

## Disclaimer

Research/educational prototype for evidence‑aware variant *prioritisation and
uncertainty triage*. **Not** a diagnostic tool and **not** intended for clinical
decision‑making or patient management.
""", encoding="utf-8")

# ---------------- docs/methods.md ----------------
(DOCS / "methods.md").write_text(r"""# Methods

## 1. Variant universe and labels

We start from the ClinVar `variant_summary.txt.gz` tab‑delimited release and keep
records whose `GeneSymbol` matches `BRCA1` or `BRCA2` (regex `\bBRCA[12]\b`). The
rsID is taken from the `RS# (dbSNP)` column and normalised to lowercase `rs…`; the
HGVS string from the `Name` column (e.g. `NM_007294.4(BRCA1):c.68_69delAG`), reduced
to the change portion after the last colon.

Labels are mapped to a binary target for supervised evaluation:

| ClinVar `ClinicalSignificance` | label |
|---|---|
| Pathogenic, Likely pathogenic, Pathogenic/Likely pathogenic | 1 |
| Benign, Likely benign, Benign/Likely benign | 0 |
| Uncertain significance, Conflicting classifications, `-`, not provided | excluded from supervised training; retained as an **uncertainty set** |

To reduce label noise we restrict the supervised pool to records whose
`ReviewStatus` is one of: *criteria provided, multiple submitters, no conflicts*;
*reviewed by expert panel*; *criteria provided, single submitter*. Records are then
deduplicated to one row per `VariationID`. This yields **16,355** high‑confidence
labelled variants (51.9% pathogenic / 48.1% benign) plus a separate VUS/conflicting
pool used only for uncertainty analysis.

## 2. Structured genomic features

Each variant is annotated via the Ensembl REST API (`/vep/human/id/`, batched POST,
JSON‑cached and resumable; `/variation/human/` for HGVS cross‑checks). From the
canonical transcript we extract:

- **consequence‑class binary flags**: `is_frameshift`, `is_stop_gained`,
  `is_splice_site` (splice acceptor/donor), `is_missense`, `is_intron`,
  `is_synonymous` (substring match on `;`‑joined `consequence_terms`);
- **SIFT** and **PolyPhen‑2** scores (present only for missense; `NaN` otherwise);
- **allele frequency** (`AF`), averaged across available population frequencies from
  colocated variants.

`NaN` SIFT/PolyPhen/AF are *informative missingness* (e.g. a frameshift has no
missense score), so the structural model is a `HistGradientBoostingClassifier`,
which handles missing values natively. The structural model is trained on the nine
features above with an 80/20 stratified split (seed 42).

## 3. Literature evidence mining

We fetch up to 5,000 PubMed records matching
`(BRCA1[Title/Abstract] OR BRCA2[Title/Abstract]) AND (pathogenic OR mutation OR variant)`
via E‑utilities (esearch → efetch, XML, batched, rate‑limited), retaining ~4,645
abstracts.

**Attribution by exact HGVS, not rsID.** Abstracts rarely print `rs…`; they print
`c.`/`p.` notations. We therefore build a token→variant dictionary from the ClinVar
`Name` column (read in 400k‑row chunks to bound RAM) using strict regexes that
capture the *full* allele change (`c.\d[\w+\-/>*_]*`, `p\.[A-Z][a-z]{2}\d+[A-Za-z0-9*/_]*`)
and **reject position‑only tokens** (e.g. bare `c.631`, trailing `c.81-`) to prevent
positional collisions where one generic sentence is credited to many variants.
Abstracts are split into sentences; a sentence is linked to a variant only if it
contains that variant's exact specific token. This precision‑first choice trades
recall for trustworthy attribution.

The mining yields **912 evidence sentences** covering **519** distinct variants
(350 labelled, 169 VUS/conflicting). After requiring *decided, non‑circular*
sentences (below), **149** labelled variants carry usable text.

## 4. Zero‑shot polarity, confidence, conflict, circularity

Each sentence is scored by a zero‑shot NLI model
(`MoritzLaurer/deberta-v3-base-zeroshot-v1`) against two hypotheses —
*"the variant is pathogenic/likely pathogenic"* and *"the variant is benign/likely
benign"* — with `multi_label=True`, giving independent `path_score` and `ben_score`.
Derived per‑sentence fields:

- `polarity` ∈ {supports_pathogenic, supports_benign, ambiguous, neutral} via a
  margin rule (|path−ben| > 0.10 and max > 0.35, else neutral/ambiguous);
- `confidence` = max(path, ben);
- `conflict_score` = 1 − |path−ben|/(path+ben) when both > 0.2, else 0;
- `circular` flag via regex for sentences that merely *restate* database calls
  (ClinVar/OncoKB/LOVD/"annotated as P/LP…"), so we do not double‑count ClinVar
  evidence into a ClinVar‑trained model.

Observed distribution: 316 supports_pathogenic, 0 supports_benign, 8 ambiguous,
588 neutral; 39/912 circular; 17 with conflict > 0.3. The **absence of benign
calls is a real, reported property** of abstract‑level zero‑shot mining on BRCA and
drives the gating design below.

## 5. Per‑variant text aggregation

For each variant we sum a per‑sentence weight
`w = confidence / variants_in_sentence` (down‑weighting sentences that list many
alleles), zeroed if `circular`. Aggregated features:
`text_path = Σ w·path_score`, `text_ben = Σ w·ben_score`,
`text_n` (# decided non‑circular sentences), `text_conflict` (max),
`text_mean_conf`, `text_n_circ`. Net signal `net_text = text_path − text_ben`.

## 6. Bayesian log‑odds fusion with gates

The fused posterior is a log‑odds update of the structural prior:
logit P(path | X, E) = logit P(path | X) + α · LR_text
LR_text = clip( has_text · net_text · mean_conf · AF_gate , −3, +3 )


- `has_text ∈ {0,1}` is the **guard**: with no decided non‑circular sentence the text
  term is exactly 0, so ~99% of variants are provably unchanged by the text channel.
- `mean_conf` scales the update by extraction confidence.
- `AF_gate = 1 / (1 + (AF / θ)²)`, θ = 1e−3, is an **ACMG‑inspired (BA1/BS1)
  population‑frequency gate**: a common variant's literature mentions are suppressed,
  preventing frequent benign polymorphisms from being falsely upgraded (worked example
  rs11571833, AF 0.0044 → gate 0.049).
- `clip(·,−3,+3)` stops one overconfident sentence from dominating.

**α selection (the key safeguard).** α is chosen on *out‑of‑fold* structural
predictions restricted to train‑fold text‑bearing variants, minimising **balanced**
log‑loss (positives and negatives each weighted 0.5) with a parsimony rule (smallest α
within +0.01 of the minimum). Ordinary log‑loss is deliberately *not* used because the
text subset is 90% pathogenic and would reward "any text ⇒ pathogenic"; the balanced
criterion yields α = 4 (interior minimum), versus α = 16 under ordinary loss.

## 7. Evaluation protocol

- **Full test set** (n=3,271): AUROC, AUPRC, Brier, balanced Brier, **Brier Skill
  Score vs the prevalence baseline**, ECE, calibration slope/intercept.
- **Matched text‑bearing slice** (test‑fold ∩ `text_n>0`, n=30): the same metrics,
  reported with **1,000‑replicate bootstrap 95% CIs** on absolute metrics and on
  fused−structural deltas. Because this slice is 90% pathogenic and tiny, the
  prevalence baseline is strong and *all* models (including structural) show negative
  absolute BSS; we therefore interpret the **delta** (fusion vs structural) and the
  calibration slope/intercept shift, not the absolute BSS, and state the subset
  degeneracy explicitly.
- **Ablations**: α = 0 (pure structural) vs fused with vs without the AF gate; AF
  threshold sensitivity (θ ∈ {2e−4 … 1e−2}); α sensitivity (descriptive).
- **Uncertainty triage**: VUS/conflicting variants (not in the supervised pool) are
  scored with the same aggregation; conflict/volume/circularity distributions are
  compared against confidently labelled variants (figure only — no pseudo‑labels).
- **Leakage controls**: variant‑level dedup; stratified split on `VariationID`; α fit
  only on train‑fold text‑bearing rows; circular sentences excluded from evidence
  strength.

Structural feature importance (permutation, AUROC drop, test fold):
`is_frameshift` 0.180, `is_synonymous` 0.109, `AF` 0.057, `is_intron` 0.015,
`is_splice_site` 0.008, `SIFT` 0.006, `is_stop_gained` 0.004, `PolyPhen` 0.000,
`is_missense` −0.000 — i.e. consequence class and population frequency dominate,
consistent with high‑confidence ClinVar labels.
""", encoding="utf-8")

# ---------------- docs/limitations.md ----------------
(DOCS / "limitations.md").write_text(r"""# Limitations and honest boundaries

These are stated up front because overclaiming here is the fastest route to rejection.
Each is also a concrete future‑work item.

1. **Sparse text coverage.** Usable decided, non‑circular literature evidence exists
   for only **149 / 16,355** high‑confidence BRCA1/2 variants (~0.9%). The fusion
   therefore cannot, by construction, move full‑set discrimination much; its value is
   on the minority of discussed variants and in *how* it handles them. Extending to
   full‑text (Europe PMC / OA articles) and to more genes would raise coverage.

2. **One‑sided text signal.** The zero‑shot extractor produced **0** benign‑supporting
   sentences. Text can only nudge toward pathogenic. We do **not** treat "no benign
   text" as benign evidence (the `has_text` guard + clipping enforce this), but a
   symmetric signal would require better benign phrasing templates or fine‑tuning.

3. **Pathogenic‑enriched, tiny matched subset.** The text‑bearing labelled subset is
   **90.6% pathogenic** and the test slice is **n = 30**. Consequently the *absolute*
   Brier Skill Score on that slice is negative for **every** model, including the
   strong structural one — a property of the degenerate subset and the prevalence
   baseline, not a failure of fusion. We report bootstrap deltas (100% favourable) and
   the calibration slope/intercept shift instead, and flag the slice‑level calibration
   parameters as unstable at this n.

4. **ClinVar labels are consensus, not gold truth.** Even after review‑status
   filtering, labels carry curator disagreement and temporal drift. We treat them as
   the best available approximation and exclude VUS/conflicting from supervised
   training rather than forcing a label.

5. **Abstracts, not full text.** Evidence is mined from PubMed abstracts only. Much
   functional/segregation detail lives in methods/results sections we did not parse,
   which caps recall and biases toward variants prominent in abstracts.

6. **Zero‑shot, not fine‑tuned, NLP.** Polarity comes from a general NLI model with
   two fixed hypotheses; it is not trained on variant‑interpretation text. The manual
   read of representative sentences (rescue‑context false positive, circular
   restatement, weak‑claim low confidence, common‑variant text trap) shows the failure
   modes a fine‑tuned biomedical classifier (e.g. PubMedBERT) would target. The
   912‑sentence file is released precisely to enable that.

7. **No temporal evaluation yet.** The current split is random at variant level. A
   prospective split (older literature → newer ClinVar reclassifications) is the
   strongest test of real‑world utility and is planned future work.

8. **Single‑gene family.** Results are BRCA1/2‑specific; consequence/frequency
   priors and literature density differ elsewhere. Generalisation requires a second
   panel (e.g. Lynch or cardiomyopathy genes) as a held‑out test.

9. **Not clinical.** This is a research/educational prototype for evidence
   prioritisation and uncertainty triage. It is **not** validated for diagnosis,
   screening, or patient management, and must not be used as such.
""", encoding="utf-8")

# ---------------- docs/results_summary.md ----------------
(DOCS / "results_summary.md").write_text(r"""# Results summary (Milestone 8D)

All numbers from the 8D run (balanced α selection, prevalence‑aware calibration).
Source tables/figures: `results/tables/*_8D.csv`, `results/figures/*_8D.png`,
plus 8A diagnostics and 8C case studies / feature importance.

## Coverage and balance (8A)

| Quantity | Value |
|---|---|
| High‑confidence labelled universe | 16,355 |
| Overall label balance | 51.9% path / 48.1% benign |
| Evidence sentences (clean) | 912 |
| Distinct variants with evidence | 519 (350 labelled, 169 VUS/conflicting) |
| Labelled variants with **decided, non‑circular** text | **149** |
| Text‑bearing labelled balance | **90.6% path / 9.4% benign** (the skew we control) |
| Circular sentences | 39 / 912 |
| Sentences with conflict > 0.3 | 17 |

## α selection (8D, balanced OOF log‑loss on 119 train text‑bearing variants)

| α | balanced log‑loss | selected? |
|---|---|---|
| 0 | 0.2901 | |
| 0.25 | 0.2786 | |
| 0.5 | 0.2686 | |
| 1 | 0.2520 | |
| 2 | 0.2293 | |
| **4** | **0.2115** | **✔ (parsimonious min)** |
| 8 | 0.2334 | |
| 16 | 0.3757 | (excluded from selection) |

Ordinary log‑loss would have chosen 16 (bias toward "any text ⇒ pathogenic"); the
balanced criterion chooses 4. This is the central methodological safeguard.

## Table 1 — Full test set (n = 3,271, prevalence 0.519)

| Model | AUROC | AUPRC | Brier | bal. Brier | BSS | ECE | slope | intercept |
|---|---|---|---|---|---|---|---|---|
| S structural | 0.9028 | 0.9005 | 0.1218 | 0.1217 | **0.512** | 0.0175 | 0.903 | 0.028 |
| F fused + AF gate | 0.9040 | 0.9018 | 0.1211 | 0.1210 | **0.515** | 0.0174 | 0.909 | 0.021 |
| F fused, no AF gate | 0.9040 | 0.9018 | 0.1211 | 0.1210 | 0.515 | 0.0174 | 0.909 | 0.021 |

Both models are well calibrated and beat the prevalence baseline strongly (BSS ≈ 0.51).
Fusion is a small, safe upgrade and never damages the structural baseline.

## Table 2 — Matched text‑bearing slice (n = 30, prevalence 0.90)

| Model | AUROC | AUPRC | Brier | bal. Brier | BSS | ECE | slope | intercept |
|---|---|---|---|---|---|---|---|---|
| S structural | 0.9938 | 0.9987 | 0.2057 | 0.1143 | −1.285 | 0.3341 | 2.81 | 11.48 |
| F fused + AF gate | 0.9877 | 0.9987 | 0.1309 | 0.0732 | −0.455 | 0.2286 | 1.46 | 4.96 |
| F fused, no AF gate | 0.9877 | 0.9987 | 0.1309 | 0.0732 | −0.455 | 0.2282 | 1.67 | 5.34 |

> **Read this correctly.** The slice is 90% pathogenic and n=30, so the trivial
> "predict 0.9" baseline (Brier ≈ 0.090) beats *every* model here — hence negative
> absolute BSS for structural **and** fused. That is a subset artefact, not a fusion
> failure. The meaningful signal is the *direction*: fusion shrinks the structural
> model's overconfidence (slope 2.81→1.46, intercept 11.48→4.96) and improves every
> calibration metric, with AUROC essentially unchanged. Slice‑level slope/intercept
> are noisy at n=30 and are reported with bootstrap CIs below.

## Bootstrap (1,000 reps) — fused − structural on the matched slice

| Metric | q2.5 | median | q97.5 | % reps favouring fusion |
|---|---|---|---|---|
| Δ AUROC | −0.029 | 0.000 | 0.000 | (never meaningfully hurt) |
| Δ Brier | −0.116 | **−0.073** | −0.037 | **100%** |
| Δ balanced Brier | −0.063 | **−0.040** | −0.021 | **100%** |
| Δ ECE | −0.167 | **−0.103** | −0.053 | **100%** |
| Δ BSS | +0.303 | **+0.850** | +3.176 | **100%** |

Fusion improves calibration on the text‑bearing subset in **100% of replicates**
across Brier, balanced Brier, ECE and BSS, while leaving discrimination intact.

## Worked example — rs11571833 (benign common polymorphism)

| | value |
|---|---|
| ClinVar label | 0 (benign) |
| AF | 0.0044 |
| net_text (path−ben) | 1.916 (4 decided sentences) |
| AF‑gate multiplier | 0.049 |
| P(structural) | 0.0016 |
| P(fused, **no** gate) | 0.0130 |
| P(fused, **+** gate) | 0.0017 |

Conservative α (=4) already contains the one‑sided text damage (0.0016→0.0130); the
ACMG BA1/BS1 frequency gate removes the residual (→0.0017, back to structural). The
gate's benefit is visible **per variant**, not in aggregate slice metrics (the slice
holds only one such benign common variant), which is exactly what the AF‑threshold
sensitivity table shows (flat Brier/BSS across θ = 2e−4…5e−3).

## Structural feature importance (permutation, AUROC drop)

`is_frameshift` 0.180 › `is_synonymous` 0.109 › `AF` 0.057 › `is_intron` 0.015 ›
`is_splice_site` 0.008 › `SIFT` 0.006 › `is_stop_gained` 0.004 › `PolyPhen` 0.000 ›
`is_missense` −0.000. Consequence class and population frequency dominate — consistent
with high‑confidence ClinVar labels and with the fact that subtle missense predictors
add little once truncating/synonymous/AF are known.

## Figures

- `fig_calibration_full_8D.png` — reliability curves, full test set (S and F nearly overlap).
- `fig_calibration_matched_8D.png` — reliability curves, matched slice (F visibly less extreme).
- `fig_alpha_sensitivity_8D.png` — AUROC/Brier/BSS vs α (full and matched).
- (8C) `fig_vus_conflict_triage.png` — conflict signal, VUS‑with‑text vs confident labels.

## One‑line takeaway

A strong structural baseline (AUROC 0.903) plus sparse, one‑sided literature, fused by
a balanced‑α, confidence/specificity/provenance/frequency‑gated Bayesian update,
yields **calibrated, interpretable, uncertainty‑aware** interpretation on the minority
of variants that literature actually discusses — without ever harming the rest.
""", encoding="utf-8")

print("Wrote:")
for p in [ROOT/"README.md", DOCS/"methods.md", DOCS/"limitations.md", DOCS/"results_summary.md"]:
    print("  ", p, f"({p.stat().st_size:,} bytes)")
print("\n✅ Milestone 9A complete. Now run the Universal Backup cell so these docs reach Drive,")
print("   then we do 9B (manuscript draft) and 9C (benchmark release + git push).")


# Universal Backup

In [ ]:
from google.colab import drive
import shutil
from pathlib import Path
import os

def universal_backup():
    print("🔄 Starting Universal Backup...")

    # 1. Mount Drive
    try:
        if not os.path.exists('/content/drive/MyDrive'):
            drive.mount('/content/drive')
        else:
            print("✅ Drive already mounted.")
    except Exception as e:
        print(f"❌ Mount failed: {e}. Trying force remount...")
        drive.mount('/content/drive', force_remount=True)

    LOCAL_ROOT = Path("/content/evidvar")
    DRIVE_ROOT = Path("/content/drive/MyDrive/EvidVar_Backup")

    # Ensure Drive root exists
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

    # 2. Walk through all local files and sync them
    synced_files = 0
    skipped_files = 0

    for item in LOCAL_ROOT.rglob("*"):
        if item.is_file():
            # Calculate relative path from ROOT
            rel_path = item.relative_to(LOCAL_ROOT)
            dst_path = DRIVE_ROOT / rel_path

            # Create parent directories in Drive if they don't exist
            dst_path.parent.mkdir(parents=True, exist_ok=True)

            # Check if file needs copying
            if dst_path.exists():
                # Compare modification times (simple heuristic)
                # Note: Timezones can sometimes mess this up in Colab/Drive,
                # so we also check size if times are close.
                src_mtime = item.stat().st_mtime
                dst_mtime = dst_path.stat().st_mtime

                # If source is newer by more than 1 second, copy
                if src_mtime > dst_mtime + 1:
                    shutil.copy(item, dst_path)
                    synced_files += 1
                else:
                    skipped_files += 1
            else:
                # File doesn't exist in Drive yet -> Copy
                shutil.copy(item, dst_path)
                synced_files += 1

    print(f"\n🎉 Backup Complete!")
    print(f"   📤 Files Copied/Updated: {synced_files}")
    print(f"   ⏭️ Files Skipped (Up-to-date): {skipped_files}")
    print(f"   📂 Location: My Drive/EvidVar_Backup")

# Run the function
universal_backup()